In [0]:
%run ../00_common/data_utils

In [0]:
consumer_table = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"
hobby_table = f"{get_env_config('golden_consumer_master_database')}.t_master_hobby"
media_table = f"{get_env_config('golden_consumer_master_database')}.t_master_emedia"
phone_table = f"{get_env_config('golden_consumer_master_database')}.t_master_phone"
address_table = f"{get_env_config('golden_consumer_master_database')}.t_master_address"
optin_table = f"{get_env_config('golden_consumer_master_database')}.t_master_optin"
crossbrand_optin_table = f"{get_env_config('golden_consumer_master_database')}.t_master_crossbrand_optin"
attributes_table = f"{get_env_config('golden_consumer_master_database')}.t_master_custom_attributes"
touchpoint_table = f"{get_env_config('golden_touchpoint_master_database')}.t_touchpoint_master"
auxiliary_table = f"{get_env_config('golden_consumer_master_database')}.t_master_auxiliary_attribute"
hair_type_table = f"{get_env_config('golden_consumer_master_database')}.t_master_hair_type"
makeup_concerns_table = f"{get_env_config('golden_consumer_master_database')}.t_master_makeup_concerns"
hair_concerns_table = f"{get_env_config('golden_consumer_master_database')}.t_master_hair_concerns"
skin_concerns_table = f"{get_env_config('golden_consumer_master_database')}.t_master_skin_concerns"
terms_table = f"{get_env_config('golden_consumer_master_database')}.t_master_terms"
consumer_group_table = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer_group"
remark_table = f"{get_env_config('golden_consumer_master_database')}.t_master_remark"
notes_table = f"{get_env_config('golden_consumer_master_database')}.t_master_notes"
dim_excludesource_table = f"{get_env_config('config_database')}.t_merge_exclude_consumer_config"
program_table = f"{get_env_config('golden_consumer_master_database')}.t_master_program"
dim_cbr_exclude_program_table = f"{get_env_config('config_database')}.t_cbr_exclude_program_code"
survive_exclude_ukey_table = f"{get_env_config('config_database')}.t_survive_exclude_ukey_config"

print(f"1. consumer_table: {consumer_table}")
print(f"2. hobby_table: {hobby_table}")
print(f"3. media_table: {media_table}")
print(f"4. phone_table: {phone_table}")
print(f"5. address_table: {address_table}")
print(f"6. optin_table: {optin_table}")
print(f"7. crossbrand_optin_table: {crossbrand_optin_table}")
print(f"8. attributes_table: {attributes_table}")
print(f"8. touchpoint_table: {touchpoint_table}")
print(f"9. auxiliary_table: {auxiliary_table}")
print(f"10. hair_type_table: {hair_type_table}")
print(f"11. makeup_concerns_table: {makeup_concerns_table}")
print(f"12. hair_concerns_table: {hair_concerns_table}")
print(f"13. skin_concerns_table: {skin_concerns_table}")
print(f"14. terms_table: {terms_table}")
print(f"15. consumer_group_table: {consumer_group_table}")
print(f"16. remark_table: {remark_table}")
print(f"17. notes_table: {notes_table}")
print(f"18. dim_excludesource_table: {dim_excludesource_table}")
print(f"19. program_table: {program_table}")
print(f"20. dim_cbr_exclude_program_table: {dim_cbr_exclude_program_table}")
print(f"21. survive_exclude_ukey_table: {survive_exclude_ukey_table}")

# 时间戳格式化
timestamp_format = "yyyy-MM-dd'T'HH:mm:ss"
# 日期格式化
date_format = "yyyy-MM-dd"

# media、phone、address排除的source、quality
EXCLUDE_SOURCE = ["undausundcbiund", "undhkgundrmkund", "undjpnund01tcsund", "undtwnundas400und", "undkorundacsund", "undidnundcbiund", "undnzlundas400und"]
VALID_QUALITY = ["vld", "und"]

# JPN 排除列表
JPN_PRIORITY_LIST = ['undjpnund01tcsund']

# JPN_TRAKUTEN source从配置表取
rakuten_sources = spark.table(dim_excludesource_table).filter(F.col("tmec_type").isin([RAKUTEN])).select("tmec_sourcesystemcode").distinct().collect()
JPN_TRAKUTEN_LIST = [row.tmec_sourcesystemcode for row in rakuten_sources]

print(f"EXCLUDE_SOURCE: {EXCLUDE_SOURCE}")
print(f"VALID_QUALITY: {VALID_QUALITY}")
print(f"JPN_PRIORITY_LIST: {JPN_PRIORITY_LIST}")
print(f"JPN_TRAKUTEN_LIST: {JPN_TRAKUTEN_LIST}")

In [0]:
def update_jpn_info(jpn_df, group_cols):

    window_core = Window.partitionBy(*group_cols)

    # ------ JPN 置空name&birth信息 ------
    # 1. 标记 priority 来源
    # 2. 窗口计算：每个组合中 priority 和非 priority 的计数
    # 3. 标记当前行是否属于混合组合（同时存在两类来源）
    # 4. 最终 need_null 条件：属于混合组合且是 priority 来源
    jpn_df = jpn_df.withColumn("p_cnt", F.sum(F.when(F.col("scon_srcs_code").isin(JPN_PRIORITY_LIST), 1).otherwise(0)).over(window_core)) \
                   .withColumn("np_cnt", F.sum(F.when(~F.col("scon_srcs_code").isin(JPN_PRIORITY_LIST), 1).otherwise(0)).over(window_core)) \
                   .withColumn("is_mixed_combo", (F.col("p_cnt") > 0) & (F.col("np_cnt") > 0)) \
                   .withColumn("need_null", F.col("is_mixed_combo") & F.col("scon_srcs_code").isin(JPN_PRIORITY_LIST))
    # 5. 需要置空的字段
    name_cols = [
        "scon_localfirstname", "scon_localmiddlename", "scon_locallastname", "scon_localfullname",
        "scon_localfirstname2", "scon_localmiddlename2", "scon_locallastname2", "scon_localfullname2",
        "scon_englishfirstname", "scon_englishmiddlename", "scon_englishlastname", "scon_englishfullname"
    ]
    birth_cols = ["scon_birthday", "scon_birthmonth", "scon_birthyear"]

    # 无条件清除 trakuten 来源的姓名字段
    for c in name_cols:
        jpn_df = jpn_df.withColumn(c, F.when(F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST), F.lit(None)).otherwise(F.col(c)))

    # 有条件清除混合组合中 priority 来源的姓名和生日
    for c in name_cols + birth_cols:
        jpn_df = jpn_df.withColumn(c, F.when(F.col("need_null"), F.lit(None)).otherwise(F.col(c)))

    # 最后删除辅助列
    jpn_df = jpn_df.drop("p_cnt", "np_cnt", "is_mixed_combo", "need_null")

    return jpn_df

In [0]:
def filter_twn_data(twn_df, group_cols):
    
    window_core = Window.partitionBy(*group_cols)
    
    # ------ TWN 过滤 ------
    # 1、窗口计算：每个组合中 onlshell 和空 class code 的计数
    # 2、检查组内是否同时存在 scon_clas_code = 'onlshell' 的记录和 scon_clas_code = '' 的记录
    # 3、如果同时存在，则删除该组内所有 scon_clas_code = 'onlshell' 的记录
    twn_df = twn_df.withColumn("on_cnt", F.sum(F.when(F.coalesce(F.col("scon_clas_code"), F.lit("")) == "onlshell", 1).otherwise(0)).over(window_core)) \
                   .withColumn("empty_cnt", F.sum(F.when(F.coalesce(F.col("scon_clas_code"), F.lit("")) == "", 1).otherwise(0)).over(window_core)) \
                   .withColumn("to_delete",
                               (F.col("on_cnt") > 0) & (F.col("empty_cnt") > 0) &
                               (F.coalesce(F.col("scon_clas_code"), F.lit("")) == "onlshell") ) \
                   .filter(~F.col("to_delete")).drop("on_cnt", "empty_cnt", "to_delete")

    return twn_df

In [0]:
def get_jpn_media_data(jpn_df, media_df, optin_df, group_cols):

    EML_TYPES = ["emlprs", "mblemlprs"]            # 需要取最佳记录的类型
    # EML_CODE_LIST = ["emlprs", "emlprf"]           # 映射为 eml 的类型
    # MBLEML_CODE_LIST = ["mblemlprs", "mblemlprf"]  # 映射为 mbleml 的类型

    comm_code_expr = F.when(F.col("scme_emdt_code").isin(["emlprs", "emlprf"]), "eml") \
                       .when(F.col("scme_emdt_code").isin(["mblemlprs", "mblemlprf"]), "mbleml") \
                       .otherwise("")
    

    # ========== 【步骤1】JOIN：一次性把media + optin + jpn关联起来 ==========
    base_df = media_df.alias("m") \
        .join(optin_df.alias("o"), 
            (F.col("m.scme_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("o.scop_mrkt_code")) &
            (F.col("o.scop_comm_code") == comm_code_expr),
            "left"
        ) \
        .join(jpn_df.alias("j"), 
            (F.col("m.scme_scon_id") == F.col("j.scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("j.scon_mrkt_code")),
            "inner"
        ) \
        .filter(~F.col("j.scon_srcs_code").isin(JPN_TRAKUTEN_LIST)) \
        .select(
            F.col("m.*"),
            *[F.col(f"j.{col}") for col in [*group_cols, "scon_id", "scon_update_dt", "scon_srcs_code", "scon_delete_flag", "is_acs_source"]],
            F.col("o.scop_comm_code").alias("optin_comm_code"),  # 原始值！不映射
            F.col("o.scop_optin_dt").alias("optin_dt"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )


    # ========== 【步骤2】拆分数据：emlprs/mblemlprs 和 其他类型 ==========
    part_eml = base_df.filter(F.col("scme_emdt_code").isin(EML_TYPES))
    part_other = base_df.filter(~F.col("scme_emdt_code").isin(EML_TYPES))


    # ========== 【步骤3】处理 emlprs/mblemlprs 部分：取每个 scme_scon_id 下的最佳记录 ==========
    # 优先级1：生成一个排序值（0,1,2,3）
    #   (1) quality_code=["vld", "und"] 且 optin_flag=1，值为 0；
    #   (2) 仅quality_code=["vld", "und"]，值为 1；
    #   (3) 仅optin_flag=1的scme_id，值为 2；
    #   (4) 其他情况值为 3
    # 优先级2：当第一优先级相同时，按 scme_id 降序排列，即取 scme_id 最大的记录
    win_best = Window.partitionBy("scme_scon_id").orderBy(
        F.when((F.col("scme_quality_code").isin(VALID_QUALITY)) & (F.col("optin_flag") == 1), 0)
        .when(F.col("scme_quality_code").isin(VALID_QUALITY), 1)
        .when(F.col("optin_flag") == 1, 2)
        .otherwise(3),
        F.col("scme_id").desc()
    )

    part_eml_best = part_eml.withColumn("rn", F.row_number().over(win_best)) \
                            .filter(F.col("rn") == 1).drop("rn")

    # 类型映射：
    # scme_emdt_code：mblemlprs -> emlprs
    # optin_comm_code：mbleml -> eml
    part_eml_best = part_eml_best.withColumn("scme_emdt_code", F.when(F.col("scme_emdt_code") == "mblemlprs", "emlprs").otherwise(F.col("scme_emdt_code"))) \
                                 .withColumn("optin_comm_code", F.when(F.col("optin_comm_code") == "mbleml", "eml").otherwise(F.col("optin_comm_code")))


    # ========== 【步骤4】合并两部分 ==========
    final_df = part_eml_best.unionByName(part_other, allowMissingColumns=True)


    # ========== 【步骤5】生成 comm_code ==========
    # final_df = final_df.withColumn("comm_code", F.when(F.col("scme_emdt_code").isin(EML_CODE_LIST), "eml")
    #                                              .when(F.col("scme_emdt_code").isin(MBLEML_CODE_LIST), "mbleml")
    #                                              .otherwise("")
    #                               ) \
    final_df = final_df.withColumn("comm_code", F.col("optin_comm_code")) \
                        .withColumn("scme_address", F.coalesce(F.col("scme_address"), F.lit("")).alias("scme_address")) \
                       .select(
                           *group_cols,
                           "is_acs_source",
                           "scon_id", "scon_update_dt", "scon_srcs_code", "scon_delete_flag",
                           "scme_id", "scme_emdt_code", "scme_sourcetimestamp",
                           "scme_address", "scme_validitycode", "scme_primary_flag",
                           "scme_quality_code", "scme_quality_desc", "comm_code",
                           F.col("optin_dt").alias("optin_date"), "optin_flag"
                       )

    return final_df

In [0]:
def calculate_overall_maxtimestamp(consumer_df, media_df, phone_df, address_df, optin_df, group_cols):
    """
    计算consumer及各contact的最大时间戳
    """

    # 1. consume主表最大时间戳（优先取不是rakuten的scon_sourcetimestamp）
    profile_ts = consumer_df.groupBy(*group_cols) \
        .agg(F.coalesce(
            F.max(F.when(~F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST), F.col("scon_sourcetimestamp"))),
            F.max("scon_sourcetimestamp")).alias("MaxSourceTimeStamp")
            )

    # 2. media最大时间戳
    media_ts = consumer_df.alias("c").join(media_df.alias("m"), 
          ((F.col("c.scon_id") == F.col("m.scme_scon_id")) & (F.col("c.scon_mrkt_code") == F.col("m.scme_mrkt_code"))), "inner") \
        .filter(~F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST)) \
        .groupBy(*group_cols) \
        .agg( F.max("scme_sourcetimestamp").alias("MaxSourceTimeStamp") )

    # 3. phone最大时间戳
    phone_ts = consumer_df.alias("c").join(phone_df.alias("p"), 
          ((F.col("c.scon_id") == F.col("p.scph_scon_id")) & (F.col("c.scon_mrkt_code") == F.col("p.scph_mrkt_code"))), "inner") \
        .filter(~F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST)) \
        .groupBy(*group_cols) \
        .agg( F.max("scph_sourcetimestamp").alias("MaxSourceTimeStamp") )

    # 4. address最大时间戳
    address_ts = consumer_df.alias("c").join(address_df.alias("a"), 
          ((F.col("c.scon_id") == F.col("a.scad_scon_id")) & (F.col("c.scon_mrkt_code") == F.col("a.scad_mrkt_code"))), "inner") \
        .filter(~F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST)) \
        .groupBy(*group_cols) \
        .agg( F.max("scad_sourcetimestamp").alias("MaxSourceTimeStamp") )

    # 5. optin最大时间戳
    optin_ts = consumer_df.alias("c").join(optin_df.alias("o"), 
          ((F.col("c.scon_id") == F.col("o.scop_scon_id")) & (F.col("c.scon_mrkt_code") == F.col("o.scop_mrkt_code"))), "inner") \
        .filter(~F.col("scon_srcs_code").isin(JPN_TRAKUTEN_LIST)) \
        .groupBy(*group_cols) \
        .agg( F.max("scop_optin_dt").alias("MaxSourceTimeStamp") )

    # 6. 合并所有时间戳
    all_ts = profile_ts.union(media_ts).union(phone_ts).union(address_ts).union(optin_ts)

    # 7. 一次 groupBy 得到每个分组的最大时间戳
    overall_max = all_ts.groupBy(*group_cols).agg(F.max("MaxSourceTimeStamp").alias("maxoveralltimestamp_contact"))

    return overall_max

In [0]:
def add_namefilledflag_by_market(df):
    """
    根据 scon_mrkt_code 动态计算 NameFilledFlag。
    特殊市场：
        - jpn：检查quality_code + 基础姓名字段 + 第二本地名
        - kor：仅检查基础姓名字段是否至少一个非空（忽略quality_code）
        其他市场（aus、hkg 等）：检查quality_code + 基础姓名字段是否全空
    """
    # quality_code条件
    quality_cond = (
        (F.coalesce(F.col("scon_englishname_quality_code"), F.lit("")) == "inv") &
        (F.coalesce(F.col("scon_localname_quality_code"), F.lit("")) == "inv") &
        (F.coalesce(F.col("scon_localname2_quality_code"), F.lit("")) == "inv")
    )

    # 基础姓名字段
    base_fields = [
        "scon_englishfirstname", "scon_englishlastname", "scon_englishfullname",
        "scon_localfirstname", "scon_locallastname", "scon_localfullname"
    ]

    # 默认逻辑
    default_all_empty = F.lit(True)
    for f in base_fields:
        default_all_empty = default_all_empty & (F.coalesce(F.col(f), F.lit("")) == "")
    default_flag = F.when(quality_cond | default_all_empty, 0).otherwise(1)

    # jpn 逻辑
    jpn_fields = base_fields + [
        "scon_localfirstname2", "scon_locallastname2", "scon_localfullname2"
    ]
    jpn_all_empty = F.lit(True)
    for f in jpn_fields:
        jpn_all_empty = jpn_all_empty & (F.coalesce(F.col(f), F.lit("")) == "")
    jpn_flag = F.when(quality_cond | jpn_all_empty, 0).otherwise(1)

    # kor 逻辑
    kor_any_nonempty = F.lit(False)
    for f in base_fields:
        kor_any_nonempty = kor_any_nonempty | (F.coalesce(F.col(f), F.lit("")) != "")
    kor_flag = F.when(kor_any_nonempty, 1).otherwise(0)

    # 根据 scon_mrkt_code 选择对应的 flag
    flag = F.when(F.col("scon_mrkt_code") == "JPN", jpn_flag) \
            .when(F.col("scon_mrkt_code") == "KOR", kor_flag) \
            .otherwise(default_flag)   # 其他所有市场（aus、hkg 等）走默认逻辑

    return df.withColumn("NameFilledFlag", flag)

In [0]:
def add_namefilledflag_l2l3(df):
    """
    根据 scon_mrkt_code 动态计算 NameFilledFlag（kor l2l3与l1不一致，其他market一致）
    特殊市场：
        - jpn：检查quality_code + 基础姓名字段 + 第二本地名
        - kor：仅检查quality_code（三个quality_code是否全为 'inv'）
        其他市场（aus、hkg 等）：检查quality_code + 基础姓名字段是否全空
    """
    # quality_code条件
    quality_cond = (
        (F.coalesce(F.col("scon_englishname_quality_code"), F.lit("")) == "inv") &
        (F.coalesce(F.col("scon_localname_quality_code"), F.lit("")) == "inv") &
        (F.coalesce(F.col("scon_localname2_quality_code"), F.lit("")) == "inv")
    )

    # 基础姓名字段
    base_fields = [
        "scon_englishfirstname", "scon_englishlastname", "scon_englishfullname",
        "scon_localfirstname", "scon_locallastname", "scon_localfullname"
    ]

    # 默认逻辑
    default_all_empty = F.lit(True)
    for f in base_fields:
        default_all_empty = default_all_empty & (F.coalesce(F.col(f), F.lit("")) == "")
    default_flag = F.when(quality_cond | default_all_empty, 0).otherwise(1)

    # jpn 逻辑
    jpn_fields = base_fields + [
        "scon_localfirstname2", "scon_locallastname2", "scon_localfullname2"
    ]
    jpn_all_empty = F.lit(True)
    for f in jpn_fields:
        jpn_all_empty = jpn_all_empty & (F.coalesce(F.col(f), F.lit("")) == "")
    jpn_flag = F.when(quality_cond | jpn_all_empty, 0).otherwise(1)

    # kor 逻辑
    kor_flag = F.when(quality_cond, 0).otherwise(1)

    # 根据 scon_mrkt_code 选择对应的 flag
    flag = F.when(F.col("scon_mrkt_code") == "JPN", jpn_flag) \
            .when(F.col("scon_mrkt_code") == "KOR", kor_flag) \
            .otherwise(default_flag)   # 其他所有市场（aus、hkg 等）走默认逻辑

    return df.withColumn("NameFilledFlag", flag)

In [0]:
def add_maxsourcetimestamp_by_market(df, group_cols):
    """
    为 DataFrame 添加 maxsourcetimestamp 列，根据scon_mrkt_code 动态选择配置。
    """
    window_core = Window.partitionBy(*group_cols)

    # market配置字典
    market_configs = {
        'HKG_KOR': {
            'exclude_class_codes': ['emlsbs', 'onlshell'],
            'name_fields': [
                "scon_englishfirstname", "scon_englishlastname", "scon_englishfullname",
                "scon_localfirstname", "scon_locallastname", "scon_localfullname"
            ]
        },
        'JPN': {
            'exclude_class_codes': ['emlsbs', 'onlshell'],
            'name_fields': [
                "scon_englishfirstname", "scon_englishlastname", "scon_englishfullname",
                "scon_localfirstname", "scon_locallastname", "scon_localfullname",
                "scon_localfirstname2", "scon_locallastname2", "scon_localfullname2"
            ]
        },
        'OTHERS': {  # 默认配置（包括 AUS 及其他所有未特别指定的市场）
            'exclude_class_codes': ['emlsbs'],
            'name_fields': [
                "scon_englishfirstname", "scon_englishlastname", "scon_englishfullname",
                "scon_localfirstname", "scon_locallastname", "scon_localfullname"
            ]
        }
    }

    # 为每个market构建对应的 maxsourcetimestamp 表达式
    market_exprs = {}
    for market, config in market_configs.items():
        # 构建姓名非空条件
        name_not_empty = F.lit(False)
        for f in config['name_fields']:
            name_not_empty = name_not_empty | (F.coalesce(F.col(f), F.lit("")) != "")

        # 构建 class code 不在排除列表的条件 or（null 视为不在排除列表）
        class_not_excluded = ~F.col("scon_clas_code").isin(config['exclude_class_codes']) | F.col("scon_clas_code").isNull()

        # 优先级条件
        # 优先级1：至少一个姓名字段非空 and class code不在排除列表 and scon_delete_flag <> 1
        # 优先级2：class code不在排除列表 and scon_delete_flag <> 1
        # 优先级3：至少一个姓名字段非空 and scon_delete_flag <> 1
        # 默认：无条件取所有记录的最大时间戳
        cond1 = name_not_empty & class_not_excluded & (F.col("scon_delete_flag") != 1)
        cond2 = class_not_excluded & (F.col("scon_delete_flag") != 1)
        cond3 = name_not_empty & (F.col("scon_delete_flag") != 1)

        max1 = F.max(F.when(cond1, F.col("scon_sourcetimestamp"))).over(window_core)
        max2 = F.max(F.when(cond2, F.col("scon_sourcetimestamp"))).over(window_core)
        max3 = F.max(F.when(cond3, F.col("scon_sourcetimestamp"))).over(window_core)
        max_all = F.max("scon_sourcetimestamp").over(window_core)

        market_exprs[market] = F.coalesce(max1, max2, max3, max_all)

    # 根据市场选择对应的组表达式
    max_ts_expr = (
        F.when(F.col("scon_mrkt_code").isin(["HKG", "KOR"]), market_exprs['HKG_KOR'])
         .when(F.col("scon_mrkt_code") == "JPN", market_exprs['JPN'])
         .otherwise(market_exprs['OTHERS'])   # 包括 AUS及其他所有市场
    )

    return df.withColumn("maxsourcetimestamp", max_ts_expr)

In [0]:
def generate_source_system_json(consumer_df, group_cols):   
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_srcs_code", "scon_sourcetimestamp", "scon_dvsn_code", "scon_consumerid", "scon_aff_code", 
                F.lit(None).alias("TerminalId"),
                (F.coalesce(F.col("scon_srcc_action"), F.lit("")) == "DELETE").alias("DeleteFlag")) \
        .distinct() \
        # .orderBy(*group_cols, "scon_srcs_code", "scon_sourcetimestamp", "scon_dvsn_code", "scon_consumerid", "scon_aff_code")

    # 构建SourceSystem结构体
    source_system_struct = F.struct(
        F.col("scon_srcs_code").alias("@Code"),
        F.date_format(F.col("scon_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scon_aff_code").alias("AffiliateCode"),
        F.col("scon_mrkt_code").alias("MarketCode"),
        F.col("scon_dvsn_code").alias("DivisionCode"),
        F.col("scon_brnd_code").alias("BrandCode"),
        F.col("scon_consumerid").alias("ConsumerId"),
        F.col("TerminalId").alias("TerminalId"),
        F.col("DeleteFlag").alias("DeleteFlag")
    )
    
    # 生成最终DataFrame
    final_df = (sconsumer_df
        .groupBy(*group_cols)
        .agg(F.collect_list(source_system_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("SourceSystem")), options={"ignoreNullFields": "false"}).alias("SourceSystemJSON")
        ))

    return final_df

In [0]:
def generate_hobby_json(consumer_df, hobby_table, group_cols):
    # 读sconsumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读hobby表
    hobby_df = spark.table(hobby_table).select("scho_scon_id","scho_mrkt_code","scho_hbby_desc")

    cond = ((sconsumer_df.scon_id == hobby_df.scho_scon_id) &
            (sconsumer_df.scon_mrkt_code == hobby_df.scho_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(hobby_df, cond, "inner") \
        .select(*group_cols, "scho_hbby_desc").distinct() \
        # .orderBy(*group_cols, "scho_hbby_desc")

    # 构建hobby结构体
    hobby_struct = F.struct(
        F.col("scho_hbby_desc").alias("HobbyDescription")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(hobby_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("Hobby")), options={"ignoreNullFields": "false"}).alias("HobbyJSON")
        ))

    return final_df

In [0]:
def generate_media_json(consumer_df, media_df, optin_df, consumer_acs_df, group_cols):
    # consumer_df: excluded acs data

    # 构建分组键
    campaign_keys = ["scon_mrkt_code", "scon_brnd_code", "scme_address", "comm_code"]
    group_cols_code_value = group_cols + ["scme_emdt_code", "scme_address"]
    group_cols_code = group_cols + ["scme_emdt_code"]

    window_campaign = Window.partitionBy(*campaign_keys)
    win_code_value = Window.partitionBy(*group_cols_code_value)
    win_code = Window.partitionBy(*group_cols_code)

    # -------------------- 1. 基础数据 --------------------
    # comm_code映射
    comm_code_expr = F.when(F.col("scme_emdt_code").isin(["emlprs", "emlprf"]), "eml") \
                       .when(F.col("scme_emdt_code").isin(["mblemlprs", "mblemlprf"]), "mbleml") \
                       .otherwise("")

    filter_optin_df = optin_df.filter(F.col("scop_comm_code").isin(["eml", "mbleml"]))
  
    # 分离市场数据
    jpn_df = consumer_df.filter(F.col("scon_mrkt_code") == "JPN")
    others_df = consumer_df.filter(~F.col("scon_mrkt_code").isin(["JPN"]))  
    
    # jpn单独处理
    base_jpn_df = get_jpn_media_data(jpn_df, media_df, filter_optin_df, group_cols)
    base_others_df = others_df.alias("c") \
        .join(media_df.alias("m"), 
            (F.col("c.scon_id") == F.col("m.scme_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("m.scme_mrkt_code")),
            "inner"
        ) \
        .join(filter_optin_df.alias("o"), 
            (F.col("m.scme_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("o.scop_mrkt_code")) &
            (F.col("o.scop_comm_code") == comm_code_expr), 
            "left"
        ) \
        .select(
            *group_cols,
            "is_acs_source",
            F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            F.col("m.scme_id"), F.col("m.scme_emdt_code"), F.col("m.scme_sourcetimestamp"),
            F.coalesce(F.col("m.scme_address"), F.lit("")).alias("scme_address"),
            F.col("m.scme_validitycode"), F.col("m.scme_primary_flag"),
            F.col("m.scme_quality_code"), F.col("m.scme_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
    base_df = base_jpn_df.union(base_others_df)

    # -------------------- 2. tadobecampaign：取adobe中最新 opt-in 状态（只取acs source） --------------------
    campaign_df = (consumer_acs_df.alias("c")
        .join(media_df.alias("m"), 
            (F.col("c.scon_id") == F.col("m.scme_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("m.scme_mrkt_code")),
            "inner"
        )
        .join(filter_optin_df.alias("o"), 
            (F.col("m.scme_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("o.scop_mrkt_code")) &
            (F.col("o.scop_comm_code") == comm_code_expr), 
            "left"
        )
        .select(
            # *group_cols,
            # "is_acs_source",
            # F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            # F.col("m.scme_id"), F.col("m.scme_emdt_code"), F.col("m.scme_sourcetimestamp"),
            F.col("scon_mrkt_code"), 
            F.col("scon_brnd_code"),
            F.coalesce(F.col("m.scme_address"), F.lit("")).alias("scme_address"),
            F.col("m.scme_validitycode"), F.col("m.scme_primary_flag"),
            F.col("m.scme_quality_code"), F.col("m.scme_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
        .filter(F.col("scme_address") != "")
        .filter(F.col("comm_code").isin(["eml", "mbleml"]))
        .withColumn("max_optin", F.max("optin_date").over(window_campaign))
        .filter(F.col("optin_date") == F.col("max_optin"))
        .select(
            *[F.col(k).alias(f"vcem_{k}") for k in campaign_keys],
            F.col("scme_quality_code").alias("vcem_eml_quality_code"),
            F.col("scme_quality_desc").alias("vcem_eml_quality_desc"),
            F.col("comm_code").alias("vcem_eml_comm_code"),
            F.col("optin_date").alias("vcem_eml_optin_dt"),
            F.col("optin_flag").alias("vcem_eml_optin_flag")
        ).dropDuplicates([f"vcem_{k}" for k in campaign_keys])
    )

    # -------------------- 3. 核心宽表：基础数据 + Campaign（不取acs source） --------------------
    join_cond = [F.col(f"base.{k}") == F.col(f"camp.vcem_{k}") for k in campaign_keys]
    join_cond.append(F.col("base.comm_code") == F.col("camp.vcem_eml_comm_code"))
    # 只取optin表optin_date<= adobe表 optin_dt的数据
    join_cond.append(F.col("base.optin_date") <= F.col("camp.vcem_eml_optin_dt")) 

    base_campaign_df = base_df.alias("base") \
        .join(campaign_df.alias("camp"), join_cond, "left_outer") \
        .select(
            *[F.col(f"base.{col}").alias(col) for col in group_cols],
            F.col("base.scon_id"),
            F.col("base.scon_update_dt"),
            F.col("base.scon_srcs_code"), F.col("base.scon_delete_flag"),
            F.col("base.scme_id"), F.col("base.scme_emdt_code"), F.col("base.scme_sourcetimestamp"),
            F.col("base.scme_address"), F.col("base.scme_validitycode"), F.col("base.scme_primary_flag"),
            F.coalesce(F.col("camp.vcem_eml_quality_code"), F.col("base.scme_quality_code")).alias("QualityCode"),
            F.coalesce(F.col("camp.vcem_eml_quality_desc"), F.col("base.scme_quality_desc")).alias("QualityDesc"),
            F.coalesce(F.col("camp.vcem_eml_comm_code"), F.col("base.comm_code")).alias("OptinCommCode"),
            F.coalesce(F.col("camp.vcem_eml_optin_dt"), F.col("base.optin_date")).alias("OptinDate"),
            F.coalesce(F.col("camp.vcem_eml_optin_flag"), F.col("base.optin_flag")).alias("OptinFlag")
        )

    # -------------------- 4. 窗口计算（MaxOptInDate、Ovl_MaxOptInDate） --------------------
    # 4.1 每个细粒度分组的MaxOptInDate（带scme_address）
    # 4.2 取细粒度OptinDate=MaxOptInDate的记录
    # 4.2 每个粗粒度分组的Ovl_MaxOptInDate（不带scme_address）
    base_maxoptin_df = base_campaign_df \
        .withColumn("MaxOptInDate",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + scon_delete_flag <> 1
                        F.max( F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), 
                                      F.col("OptinDate"))).over(win_code_value),
                        # 优先级2：仅排除指定srcs_code
                        F.max(F.when( ~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE),F.col("OptinDate"))).over(win_code_value),
                        # 优先级3：仅scon_delete_flag <> 1
                        F.max(F.when(F.col("scon_delete_flag") != 1,F.col("OptinDate"))).over(win_code_value),
                        # 默认：全部记录的最大值
                        F.max("OptinDate").over(win_code_value)
                    )) \
        .filter(F.col("OptinDate") == F.col("MaxOptInDate")) \
        .withColumn("Ovl_MaxOptInDate",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + QualityCode=["vld", "und"] + OptinFlag=1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1),
                                     F.col("OptinDate"))).over(win_code),
                        # 优先级2：排除指定srcs_code + QualityCode=["vld", "und"]
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY), 
                                     F.col("OptinDate"))).over(win_code),
                        # 优先级3：排除指定srcs_code + OptinFlag=1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) &(F.col("OptinFlag") == 1),F.col("OptinDate"))).over(win_code),
                        # 优先级4：仅排除指定srcs_code
                        F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE),F.col("OptinDate"))).over(win_code),
                        # 优先级5：QualityCode=["vld", "und"] + OptinFlag=1
                        F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY) &(F.col("OptinFlag") == 1),F.col("OptinDate"))).over(win_code),
                        # 优先级6：仅QualityCode=["vld", "und"]
                        F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY),F.col("OptinDate"))).over(win_code),
                        # 优先级7：仅OptinFlag=1
                        F.max(F.when(F.col("OptinFlag") == 1,F.col("OptinDate"))).over(win_code),
                        # 默认：全部记录的最大值
                        F.max("OptinDate").over(win_code)
                    ))
    
    # -------------------- 5. 分别计算 有Ovl_MaxOptInDate和无Ovl_MaxOptInDate 的最大时间戳 --------------------
    # ovl有记录的组：取 OptinDate = Ovl_MaxOptInDate 的记录中的最大时间戳
    max_ts_ovl_df = base_maxoptin_df \
                     .filter(F.col("Ovl_MaxOptInDate").isNotNull()) \
                     .filter(F.col("OptinDate") == F.col("Ovl_MaxOptInDate")) \
                     .groupBy(*group_cols_code) \
                     .agg(F.max("scme_sourcetimestamp").alias("maxtimestamp"))

    # ovl无记录的组：过滤出无ovl的组（Ovl_MaxOptInDate为null），按另一套优先级取最大时间戳
    # max_ts_no_ovl_df = base_maxoptin_df.filter(F.col("Ovl_MaxOptInDate").isNull()) \
    max_ts_no_ovl_df = base_campaign_df.join(max_ts_ovl_df, group_cols_code, "left_anti") \
        .withColumn("maxtimestamp",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY),
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级2：排除指定srcs_code + scon_delete_flag <> 1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), 
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级3：scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                        F.max(F.when( (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), 
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级4：仅scon_delete_flag <> 1
                        F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scme_sourcetimestamp"))).over(win_code),
                        # 默认：全部记录的最大值
                        F.max("scme_sourcetimestamp").over(win_code)
        )).select(*group_cols_code, "maxtimestamp").distinct()

    # 合并两部分最大时间戳
    all_max_ts = max_ts_ovl_df.union(max_ts_no_ovl_df)

    # -------------------- 6. 根据最大时间戳获取最大scon_id --------------------
    # 定义窗口：按分组分区，排序规则
    #   - 优先级1：只保留时间戳等于 maxtimestamp 的记录（即筛选出该分组内最新的一批记录）
    #   - 优先级2：在这些最新记录中，先按scon_update_dt降序，再按scon_id降序取第一条
    win_final = Window.partitionBy(*group_cols_code).orderBy(
        F.when(F.col("scme_sourcetimestamp") == F.col("maxtimestamp"), 0).otherwise(1).asc(),
        F.col("scon_update_dt").desc(),
        F.col("scon_id").desc(),
        F.col("scme_id").desc(),
        F.col("OptinCommCode")
        )
    
    base_with_ts_df = base_campaign_df.join(all_max_ts, group_cols_code, "left") \
    .withColumn("rn", F.row_number().over(win_final)) \
    .filter(F.col("rn") == 1)

    # -------------------- 7. 输出两个df --------------------
    # 后面生成optin json时使用
    bDerivedEmailOptin_df = base_with_ts_df.select(
        *group_cols_code,
        F.col("scon_id").alias("Maxsconid"),
        F.col("OptinCommCode"),
        F.col("OptinDate"),
        F.col("OptinFlag")
    ).distinct()
    
    # 生成email json时使用
    json_df = base_with_ts_df
    # .dropDuplicates(group_cols_code)
    # .orderBy(group_cols_code)

    # 构建media结构体
    media_struct = F.struct(
        F.col("scme_emdt_code").alias("@TypeCode"),
        F.date_format(F.col("scme_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scme_address").alias("Address"),
        F.col("scme_primary_flag").alias("PrimaryFlag"),
        F.col("QualityCode").alias("DataQualityCode"),
        F.col("QualityDesc").alias("DataQualityDescription")
    )
    
    # 生成最终DataFrame
    final_df = (json_df
        .groupBy(*group_cols)
        .agg( F.collect_list(media_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("EMedia")), options={"ignoreNullFields": "false"}).alias("EMediaJSON")
        ))

    return bDerivedEmailOptin_df, final_df

In [0]:
def generate_media_base_ts(consumer_df, media_df, optin_df, consumer_acs_df, group_cols):
    # consumer_df: excluded acs data

    # 构建分组键
    campaign_keys = ["scon_mrkt_code", "scon_brnd_code", "scme_address", "comm_code"]
    group_cols_code_value = group_cols + ["scme_emdt_code", "scme_address"]
    group_cols_code = group_cols + ["scme_emdt_code"]

    window_campaign = Window.partitionBy(*campaign_keys)
    win_code_value = Window.partitionBy(*group_cols_code_value)
    win_code = Window.partitionBy(*group_cols_code)

    # -------------------- 1. 基础数据 --------------------
    # comm_code映射
    comm_code_expr = F.when(F.col("scme_emdt_code").isin(["emlprs", "emlprf"]), "eml") \
                       .when(F.col("scme_emdt_code").isin(["mblemlprs", "mblemlprf"]), "mbleml") \
                       .otherwise("")

    filter_optin_df = optin_df.filter(F.col("scop_comm_code").isin(["eml", "mbleml"]))
  
    # 分离市场数据
    jpn_df = consumer_df.filter(F.col("scon_mrkt_code") == "JPN")
    others_df = consumer_df.filter(~F.col("scon_mrkt_code").isin(["JPN"]))  
    
    # jpn单独处理
    base_jpn_df = get_jpn_media_data(jpn_df, media_df, filter_optin_df, group_cols)
    base_others_df = others_df.alias("c") \
        .join(media_df.alias("m"), 
            (F.col("c.scon_id") == F.col("m.scme_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("m.scme_mrkt_code")),
            "inner"
        ) \
        .join(filter_optin_df.alias("o"), 
            (F.col("m.scme_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("o.scop_mrkt_code")) &
            (F.col("o.scop_comm_code") == comm_code_expr), 
            "left"
        ) \
        .select(
            *group_cols,
            "is_acs_source",
            F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            F.col("m.scme_id"), F.col("m.scme_emdt_code"), F.col("m.scme_sourcetimestamp"),
            F.coalesce(F.col("m.scme_address"), F.lit("")).alias("scme_address"),
            F.col("m.scme_validitycode"), F.col("m.scme_primary_flag"),
            F.col("m.scme_quality_code"), F.col("m.scme_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
    base_df = base_jpn_df.union(base_others_df)

    # -------------------- 2. tadobecampaign：取adobe中最新 opt-in 状态（只取acs source） --------------------
    campaign_df = (consumer_acs_df.alias("c")
        .join(media_df.alias("m"), 
            (F.col("c.scon_id") == F.col("m.scme_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("m.scme_mrkt_code")),
            "inner"
        )
        .join(filter_optin_df.alias("o"), 
            (F.col("m.scme_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("m.scme_mrkt_code") == F.col("o.scop_mrkt_code")) &
            (F.col("o.scop_comm_code") == comm_code_expr), 
            "left"
        )
        .select(
            # *group_cols,
            # "is_acs_source",
            # F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            # F.col("m.scme_id"), F.col("m.scme_emdt_code"), F.col("m.scme_sourcetimestamp"),
            F.col("scon_mrkt_code"), 
            F.col("scon_brnd_code"),
            F.coalesce(F.col("m.scme_address"), F.lit("")).alias("scme_address"),
            F.col("m.scme_validitycode"), F.col("m.scme_primary_flag"),
            F.col("m.scme_quality_code"), F.col("m.scme_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
        .filter(F.col("scme_address") != "")
        .filter(F.col("comm_code").isin(["eml", "mbleml"]))
        .withColumn("max_optin", F.max("optin_date").over(window_campaign))
        .filter(F.col("optin_date") == F.col("max_optin"))
        .select(
            *[F.col(k).alias(f"vcem_{k}") for k in campaign_keys],
            F.col("scme_quality_code").alias("vcem_eml_quality_code"),
            F.col("scme_quality_desc").alias("vcem_eml_quality_desc"),
            F.col("comm_code").alias("vcem_eml_comm_code"),
            F.col("optin_date").alias("vcem_eml_optin_dt"),
            F.col("optin_flag").alias("vcem_eml_optin_flag")
        ).dropDuplicates([f"vcem_{k}" for k in campaign_keys])
    )

    # -------------------- 3. 核心宽表：基础数据 + Campaign（不取acs source） --------------------
    join_cond = [F.col(f"base.{k}") == F.col(f"camp.vcem_{k}") for k in campaign_keys]
    join_cond.append(F.col("base.comm_code") == F.col("camp.vcem_eml_comm_code"))
    # 只取optin表optin_date<= adobe表 optin_dt的数据
    join_cond.append(F.col("base.optin_date") <= F.col("camp.vcem_eml_optin_dt")) 

    base_campaign_df = base_df.alias("base") \
        .join(campaign_df.alias("camp"), join_cond, "left_outer") \
        .select(
            *[F.col(f"base.{col}").alias(col) for col in group_cols],
            F.col("base.scon_id"),
            F.col("base.scon_update_dt"),
            F.col("base.scon_srcs_code"), F.col("base.scon_delete_flag"),
            F.col("base.scme_id"), F.col("base.scme_emdt_code"), F.col("base.scme_sourcetimestamp"),
            F.col("base.scme_address"), F.col("base.scme_validitycode"), F.col("base.scme_primary_flag"),
            F.coalesce(F.col("camp.vcem_eml_quality_code"), F.col("base.scme_quality_code")).alias("QualityCode"),
            F.coalesce(F.col("camp.vcem_eml_quality_desc"), F.col("base.scme_quality_desc")).alias("QualityDesc"),
            F.coalesce(F.col("camp.vcem_eml_comm_code"), F.col("base.comm_code")).alias("OptinCommCode"),
            F.coalesce(F.col("camp.vcem_eml_optin_dt"), F.col("base.optin_date")).alias("OptinDate"),
            F.coalesce(F.col("camp.vcem_eml_optin_flag"), F.col("base.optin_flag")).alias("OptinFlag")
        )

    # -------------------- 4. 窗口计算（MaxOptInDate、Ovl_MaxOptInDate） --------------------
    # 4.1 每个细粒度分组的MaxOptInDate（带scme_address）
    # 4.2 取细粒度OptinDate=MaxOptInDate的记录
    # 4.2 每个粗粒度分组的Ovl_MaxOptInDate（不带scme_address）
    base_maxoptin_df = base_campaign_df \
        .withColumn("MaxOptInDate",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + scon_delete_flag <> 1
                        F.max( F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), 
                                      F.col("OptinDate"))).over(win_code_value),
                        # 优先级2：仅排除指定srcs_code
                        F.max(F.when( ~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE),F.col("OptinDate"))).over(win_code_value),
                        # 优先级3：仅scon_delete_flag <> 1
                        F.max(F.when(F.col("scon_delete_flag") != 1,F.col("OptinDate"))).over(win_code_value),
                        # 默认：全部记录的最大值
                        F.max("OptinDate").over(win_code_value)
                    )) \
        .filter(F.col("OptinDate") == F.col("MaxOptInDate")) \
        .withColumn("Ovl_MaxOptInDate",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + QualityCode=["vld", "und"] + OptinFlag=1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1),
                                     F.col("OptinDate"))).over(win_code),
                        # 优先级2：排除指定srcs_code + QualityCode=["vld", "und"]
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY), 
                                     F.col("OptinDate"))).over(win_code),
                        # 优先级3：排除指定srcs_code + OptinFlag=1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) &(F.col("OptinFlag") == 1),F.col("OptinDate"))).over(win_code),
                        # 优先级4：仅排除指定srcs_code
                        F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE),F.col("OptinDate"))).over(win_code),
                        # 优先级5：QualityCode=["vld", "und"] + OptinFlag=1
                        F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY) &(F.col("OptinFlag") == 1),F.col("OptinDate"))).over(win_code),
                        # 优先级6：仅QualityCode=["vld", "und"]
                        F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY),F.col("OptinDate"))).over(win_code),
                        # 优先级7：仅OptinFlag=1
                        F.max(F.when(F.col("OptinFlag") == 1,F.col("OptinDate"))).over(win_code),
                        # 默认：全部记录的最大值
                        F.max("OptinDate").over(win_code)
                    ))
    
    # -------------------- 5. 分别计算 有Ovl_MaxOptInDate和无Ovl_MaxOptInDate 的最大时间戳 --------------------
    # ovl有记录的组：取 OptinDate = Ovl_MaxOptInDate 的记录中的最大时间戳
    max_ts_ovl_df = base_maxoptin_df \
                     .filter(F.col("Ovl_MaxOptInDate").isNotNull()) \
                     .filter(F.col("OptinDate") == F.col("Ovl_MaxOptInDate")) \
                     .groupBy(*group_cols_code) \
                     .agg(F.max("scme_sourcetimestamp").alias("maxtimestamp"))

    # ovl无记录的组：过滤出无ovl的组（Ovl_MaxOptInDate为null），按另一套优先级取最大时间戳
    # max_ts_no_ovl_df = base_maxoptin_df.filter(F.col("Ovl_MaxOptInDate").isNull()) \
    max_ts_no_ovl_df = base_campaign_df.join(max_ts_ovl_df, group_cols_code, "left_anti") \
        .withColumn("maxtimestamp",
                    F.coalesce(
                        # 优先级1：排除指定srcs_code + scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY),
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级2：排除指定srcs_code + scon_delete_flag <> 1
                        F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), 
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级3：scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                        F.max(F.when( (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), 
                                     F.col("scme_sourcetimestamp"))).over(win_code),
                        # 优先级4：仅scon_delete_flag <> 1
                        F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scme_sourcetimestamp"))).over(win_code),
                        # 默认：全部记录的最大值
                        F.max("scme_sourcetimestamp").over(win_code)
        )).select(*group_cols_code, "maxtimestamp").distinct()

    # 合并两部分最大时间戳
    all_max_ts = max_ts_ovl_df.union(max_ts_no_ovl_df)

    # -------------------- 6. 根据最大时间戳获取最大scon_id --------------------
    # 定义窗口：按分组分区，排序规则
    #   - 优先级1：只保留时间戳等于 maxtimestamp 的记录（即筛选出该分组内最新的一批记录）
    #   - 优先级2：在这些最新记录中，先按scon_update_dt降序，再按scon_id降序取第一条
    win_final = Window.partitionBy(*group_cols_code).orderBy(
        F.when(F.col("scme_sourcetimestamp") == F.col("maxtimestamp"), 0).otherwise(1).asc(),
        F.col("scon_update_dt").desc(),
        F.col("scon_id").desc(),
        F.col("scme_id").desc()
        )
    
    base_with_ts_df = base_campaign_df.join(all_max_ts, group_cols_code, "left") \
    .withColumn("rn", F.row_number().over(win_final)) \
    .filter(F.col("rn") == 1)

    # -------------------- 7. 输出两个df --------------------
    # 后面生成optin json时使用
    # bDerivedEmailOptin_df = base_with_ts_df.select(
    #     *group_cols_code,
    #     F.col("scon_id").alias("Maxsconid"),
    #     F.col("OptinCommCode"),
    #     F.col("OptinDate"),
    #     F.col("OptinFlag")
    # ).distinct()
    
    # 生成email json时使用
    # json_df = base_with_ts_df.dropDuplicates(group_cols_code)

    # # 构建media结构体
    # media_struct = F.struct(
    #     F.col("scme_emdt_code").alias("@TypeCode"),
    #     F.date_format(F.col("scme_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
    #     F.col("scme_address").alias("Address"),
    #     F.col("scme_primary_flag").alias("PrimaryFlag"),
    #     F.col("QualityCode").alias("DataQualityCode"),
    #     F.col("QualityDesc").alias("DataQualityDescription")
    # )
    
    # # 生成最终DataFrame
    # final_df = (json_df
    #     .groupBy(*group_cols)
    #     .agg( F.collect_list(media_struct).alias("item_list"))
    #     .select(
    #         *group_cols,
    #         F.to_json(F.struct(F.col("item_list").alias("EMedia")), options={"ignoreNullFields": "false"}).alias("EMediaJSON")
    #     ))

    # return bDerivedEmailOptin_df, final_df
    return base_with_ts_df

def generate_media_optin(base_with_ts_df, group_cols):
    group_cols_code = group_cols + ["scme_emdt_code"]

    bDerivedEmailOptin_df = base_with_ts_df.select(
        *group_cols_code,
        F.col("scon_id").alias("Maxsconid"),
        F.col("OptinCommCode"),
        F.col("OptinDate"),
        F.col("OptinFlag")
    ).distinct()
    
    return bDerivedEmailOptin_df


def new_generate_media_json(base_with_ts_df, group_cols):
    group_cols_code = group_cols + ["scme_emdt_code"]

    # 生成email json时使用
    # json_df = base_with_ts_df.dropDuplicates(group_cols_code)

    # 构建media结构体
    media_struct = F.struct(
        F.col("scme_emdt_code").alias("@TypeCode"),
        F.date_format(F.col("scme_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scme_address").alias("Address"),
        F.col("scme_primary_flag").alias("PrimaryFlag"),
        F.col("QualityCode").alias("DataQualityCode"),
        F.col("QualityDesc").alias("DataQualityDescription")
    )
    
    # 生成最终DataFrame
    final_df = (base_with_ts_df
        .groupBy(*group_cols)
        .agg( F.collect_list(media_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("EMedia")), options={"ignoreNullFields": "false"}).alias("EMediaJSON")
        ))

    return final_df

In [0]:
def generate_phone_json(consumer_df, phone_df, optin_df, consumer_acs_df, group_cols):
    # consumer_df: excluded acs data

    # 构建分组键
    campaign_keys = ["scon_mrkt_code", "scon_brnd_code", "scph_phonenumber", "optin_comm_code"]
    group_cols_code_value = group_cols + ["scph_phnt_code", "scph_phonenumber"]
    group_cols_code = group_cols + ["scph_phnt_code"]

    window_campaign = Window.partitionBy(*campaign_keys)
    win_code_value = Window.partitionBy(*group_cols_code_value)
    win_code = Window.partitionBy(*group_cols_code)


    filter_optin_df = optin_df.filter(F.col("scop_comm_code").isin(["sms", "mms"]))
    # -------------------- 1. 基础数据 --------------------
    base_df = consumer_df.alias("c") \
        .join( phone_df.alias("p"), 
            (F.col("c.scon_id") == F.col("p.scph_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("p.scph_mrkt_code")),
            "inner"
        ) \
        .join(filter_optin_df.alias("o"),
            (F.col("p.scph_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("p.scph_mrkt_code") == F.col("o.scop_mrkt_code")) &
            F.col("o.scop_comm_code").isin(["sms", "mms"]) &
            # 如果是 KOR，电话类型无条件通过；其他market必须是指定类型
            ( (F.col("c.scon_mrkt_code") == "KOR") | ((F.col("c.scon_mrkt_code") != "KOR") & F.col("p.scph_phnt_code").isin(["mblprs", "mblprf"])) ), 
            "left"
        ) \
        .select(
            *group_cols,
            F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            F.col("p.scph_id"),
            F.col("p.scph_phnt_code"),
            F.col("p.scph_sourcetimestamp"),
            F.col("p.scph_phonecountrycode"),
            F.coalesce(F.col("p.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.col("p.scph_validitycode"),
            F.col("p.scph_primary_flag"),
            F.col("p.scph_quality_code"),
            F.col("p.scph_quality_desc"),
            F.col("o.scop_comm_code").alias("optin_comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )

    # -------------------- 2. tadobecampaign：取adobe中最新 opt-in 状态（只取acs source） --------------------
        # 组合条件：对于 AUS/JPN 市场，必须同时满足来源、电话号码和 quality_desc 条件；
        # 对于其他市场，无条件接受（即 OR 分支为 True）
    campaign_df = (consumer_acs_df.alias("c")
        .join( phone_df.alias("p"), 
            (F.col("c.scon_id") == F.col("p.scph_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("p.scph_mrkt_code")),
            "inner"
        )
        .join(filter_optin_df.alias("o"),
            (F.col("p.scph_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("p.scph_mrkt_code") == F.col("o.scop_mrkt_code")) &
            F.col("o.scop_comm_code").isin(["sms", "mms"]) &
            # 如果是 KOR，电话类型无条件通过；其他market必须是指定类型
            ( (F.col("c.scon_mrkt_code") == "KOR") | ((F.col("c.scon_mrkt_code") != "KOR") & F.col("p.scph_phnt_code").isin(["mblprs", "mblprf"])) ), 
            "left"
        )
        .select(
            # *group_cols,
            # F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            # F.col("p.scph_id"),
            F.col("scon_mrkt_code"), 
            F.col("scon_brnd_code"),
            F.col("p.scph_phnt_code"),
            F.col("p.scph_sourcetimestamp"),
            F.col("p.scph_phonecountrycode"),
            F.coalesce(F.col("p.scph_phonenumber"), F.lit("")).alias("scph_phonenumber"),
            F.col("p.scph_validitycode"),
            F.col("p.scph_primary_flag"),
            F.col("p.scph_quality_code"),
            F.col("p.scph_quality_desc"),
            F.col("o.scop_comm_code").alias("optin_comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
        .filter(
            F.col("optin_comm_code").isin(["sms", "mms"]) &
            # 如果是 KOR，电话类型无条件通过；其他market必须是指定类型
            ( (F.col("scon_mrkt_code") == "KOR") | ((F.col("scon_mrkt_code") != "KOR") & F.col("scph_phnt_code").isin(["mblprs", "mblprf"])) ))
        .filter(
            (F.col("scon_mrkt_code").isin(["AUS", "JPN"]) & (F.col("scph_quality_desc") != "Invalid") & (F.length(F.col("scph_phonenumber")) > 2)) |
            (~F.col("scon_mrkt_code").isin(["AUS", "JPN"]))
        )
        .withColumn("max_optin", F.max("optin_date").over(window_campaign))
        .filter(F.col("optin_date") == F.col("max_optin"))
        .select(
            *[F.col(k).alias(f"vcph_{k}") for k in campaign_keys],
            F.col("scph_quality_code").alias("vcph_phn_quality_code"),
            F.col("scph_quality_desc").alias("vcph_phn_quality_desc"),
            F.col("optin_comm_code").alias("vcph_phn_comm_code"),
            F.col("optin_date").alias("vcph_phn_optin_dt"),
            F.col("optin_flag").alias("vcph_phn_optin_flag")
        ).dropDuplicates([f"vcph_{k}" for k in campaign_keys])
    )

    # -------------------- 3. 核心宽表：基础数据 + Campaign（不取acs source） --------------------
    join_cond = [F.col(f"base.{k}") == F.col(f"camp.vcph_{k}") for k in campaign_keys]
    join_cond.append(F.col("base.optin_comm_code") == F.col("camp.vcph_phn_comm_code"))
    # 只取optin表optin_date<= adobe表 optin_dt的数据
    join_cond.append(F.col("base.optin_date") <= F.col("camp.vcph_phn_optin_dt"))

    base_campaign_df = base_df.alias("base") \
        .join(campaign_df.alias("camp"), join_cond, "left_outer") \
        .select(
            *[F.col(f"base.{col}").alias(col) for col in group_cols],
            F.col("base.scon_id"),
            F.col("base.scon_update_dt"),
            F.col("base.scon_srcs_code"), F.col("base.scon_delete_flag"),
            F.col("base.scph_id"),
            F.col("base.scph_phnt_code"),
            F.col("base.scph_sourcetimestamp"),
            F.col("base.scph_phonecountrycode"),
            F.col("base.scph_phonenumber"),
            F.col("base.scph_validitycode"),
            F.col("base.scph_primary_flag"),
            F.coalesce(F.col("camp.vcph_phn_quality_code"), F.col("base.scph_quality_code")).alias("QualityCode"),
            F.coalesce(F.col("camp.vcph_phn_quality_desc"), F.col("base.scph_quality_desc")).alias("QualityDesc"),
            F.coalesce(F.col("camp.vcph_phn_comm_code"), F.col("base.optin_comm_code")).alias("OptinCommCode"),
            F.coalesce(F.col("camp.vcph_phn_optin_dt"), F.col("base.optin_date")).alias("OptinDate"),
            F.coalesce(F.col("camp.vcph_phn_optin_flag"), F.col("base.optin_flag")).alias("OptinFlag")
        )

    # -------------------- 4. 窗口计算（MaxOptInDate、Ovl_MaxOptInDate） --------------------
    base_maxoptin_df = base_campaign_df \
        .withColumn("MaxOptInDate",
            F.coalesce(
                # 优先级1：排除指定srcs_code + scon_delete_flag <> 1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), F.col("OptinDate"))).over(win_code_value),
                # 优先级2：仅scon_delete_flag <> 1
                F.max(F.when(F.col("scon_delete_flag") != 1, F.col("OptinDate"))).over(win_code_value),
                # 优先级3：仅排除指定srcs_code
                F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE), F.col("OptinDate"))).over(win_code_value),
                # 默认：全部记录的最大值
                F.max("OptinDate").over(win_code_value)
            )) \
        .filter(F.col("OptinDate") == F.col("MaxOptInDate")) \
        .withColumn("Ovl_MaxOptInDate",
            F.coalesce(
                # 优先级1：排除指定srcs_code + QualityCode=["vld", "und"] + OptinFlag=1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级2：排除指定srcs_code + QualityCode=["vld", "und"]
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY), F.col("OptinDate"))).over(win_code),
                # 优先级3：排除指定srcs_code + OptinFlag=1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级4：仅排除指定srcs_code
                F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE), F.col("OptinDate"))).over(win_code),
                # 优先级5：QualityCode=["vld", "und"] + OptinFlag=1
                F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级6：仅QualityCode=["vld", "und"]
                F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY), F.col("OptinDate"))).over(win_code),
                # 优先级7：仅OptinFlag=1
                F.max(F.when(F.col("OptinFlag") == 1, F.col("OptinDate"))).over(win_code),
                # 默认：全部记录的最大值
                F.max("OptinDate").over(win_code)
            ))

    # -------------------- 5. 分别计算 有Ovl_MaxOptInDate和无Ovl_MaxOptInDate 的最大时间戳 --------------------
    # ovl有记录的组：取 OptinDate = Ovl_MaxOptInDate 的记录中的最大时间戳
    max_ts_ovl_df = base_maxoptin_df \
        .filter(F.col("Ovl_MaxOptInDate").isNotNull()) \
        .filter(F.col("OptinDate") == F.col("Ovl_MaxOptInDate")) \
        .groupBy(*group_cols_code) \
        .agg(F.max("scph_sourcetimestamp").alias("maxtimestamp"))

    # ovl无记录的组：过滤出无ovl的组（Ovl_MaxOptInDate为null），按另一套优先级取最大时间戳
    # max_ts_no_ovl_df = base_maxoptin_df.filter(F.col("Ovl_MaxOptInDate").isNull()) \
    max_ts_no_ovl_df = base_campaign_df.join(max_ts_ovl_df, group_cols_code, "left_anti") \
        .withColumn("maxtimestamp",
            F.coalesce(
                # 优先级1：排除指定srcs_code + scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), F.col("scph_sourcetimestamp"))).over(win_code),
                # 优先级2：排除指定srcs_code + scon_delete_flag <> 1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), F.col("scph_sourcetimestamp"))).over(win_code),
                # 优先级3：scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                F.max(F.when((F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), F.col("scph_sourcetimestamp"))).over(win_code),
                # 优先级4：仅scon_delete_flag <> 1
                F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scph_sourcetimestamp"))).over(win_code),
                # 默认：全部记录的最大值
                F.max("scph_sourcetimestamp").over(win_code)
            )
        ).select(*group_cols_code, "maxtimestamp").distinct()

    # 合并两部分最大时间戳
    all_max_ts = max_ts_ovl_df.union(max_ts_no_ovl_df)

    # -------------------- 6. 根据最大时间戳获取最大scon_id --------------------
    # 定义窗口：按分组分区，排序规则
    #   - 优先级1：只保留时间戳等于 maxtimestamp 的记录（即筛选出该分组内最新的一批记录）
    #   - 优先级2：在这些最新记录中，先按scon_update_dt降序，再按scon_id降序取第一条
    win_final = Window.partitionBy(*group_cols_code).orderBy(
        F.when(F.col("scph_sourcetimestamp") == F.col("maxtimestamp"), 0).otherwise(1).asc(),
        F.col("scon_update_dt").desc(),
        F.col("scon_id").desc(),
        F.col("scph_id").desc(),
        F.col("OptinCommCode")
    )

    base_with_ts_df = base_campaign_df.join(all_max_ts, group_cols_code, "left") \
        .withColumn("rn", F.row_number().over(win_final)) \
        .filter(F.col("rn") == 1)

    # -------------------- 7. 输出两个df --------------------
    # 后面生成optin json时使用
    bDerivedPhoneOptin_df = base_with_ts_df.select(
            *group_cols_code,
            F.col("scon_id").alias("Maxsconid"),
            F.col("OptinCommCode"),
            F.col("OptinDate"),
            F.col("OptinFlag")
        ).distinct()
    
    # 生成phone json时使用
    json_df = base_with_ts_df
    # .dropDuplicates(group_cols_code)
    # .orderBy(group_cols_code)
    
    # 构建phone结构体
    phone_struct = F.struct(
        F.col("scph_phnt_code").alias("@TypeCode"),
        F.date_format(F.col("scph_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.lit(None).alias("CountryCode_ISO3"),
        F.when(F.trim(F.coalesce(F.col("scph_phonecountrycode"), F.lit(""))) == "", F.lit(None)).otherwise(F.col("scph_phonecountrycode")).alias("PhoneCountryCode"),
        F.lit(None).alias("RegionalCode"),
        F.lit(None).alias("Extension"),
        F.col("scph_phonenumber").alias("PhoneNumber"),
        F.lit(None).alias("RegPhoneTypeCode"),
        F.lit(None).alias("Registrar"),
        F.col("QualityCode").alias("DataQualityCode"),
        F.col("QualityDesc").alias("DataQualityDescription")
    )
    
    # 生成最终DataFrame
    final_df = json_df \
        .groupBy(*group_cols) \
        .agg(F.collect_list(phone_struct).alias("item_list")) \
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("Phone")), options={"ignoreNullFields": "false"}).alias("PhoneJSON")
        )

    return bDerivedPhoneOptin_df, final_df

In [0]:
def generate_address_json(consumer_df, address_df, optin_df, consumer_acs_df, group_cols):
    # consumer_df: excluded acs data

    # 构建分组键
    campaign_keys = ["scon_mrkt_code", "scon_brnd_code", "scad_address1", "comm_code"]
    group_cols_code_value = group_cols + ["scad_addt_code", "scad_address1"]
    group_cols_code = group_cols + ["scad_addt_code"]

    # 窗口定义
    window_campaign = Window.partitionBy(*campaign_keys)
    win_code_value = Window.partitionBy(*group_cols_code_value)
    win_code = Window.partitionBy(*group_cols_code)

    # -------------------- 1. 基础数据 --------------------
    # comm_code_expr = F.lit("drcml")
    filter_optin_df = optin_df.filter(F.col("scop_comm_code") == F.lit("drcml"))

    base_df = consumer_df.alias("c") \
        .join(address_df.alias("a"), 
            (F.col("c.scon_id") == F.col("a.scad_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("a.scad_mrkt_code")),
            "inner"
        ) \
        .join(filter_optin_df.alias("o"),
            (F.col("a.scad_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("a.scad_mrkt_code") == F.col("o.scop_mrkt_code")) 
            # & (F.col("o.scop_comm_code") == comm_code_expr)
            ,
            "left"
        ) \
        .select(
            *group_cols,
            F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            F.col("a.scad_id"),
            F.col("a.scad_addt_code"),
            F.col("a.scad_sourcetimestamp"),
            F.col("a.scad_address1"),
            F.col("a.scad_address2"),
            F.col("a.scad_address3"),
            F.col("a.scad_city_localdesc"),
            F.col("a.scad_prvn_localdesc"),
            F.col("a.scad_cntr_isoalpha3code"),
            F.col("a.scad_postalcode"),
            F.col("a.scad_validitycode"),
            F.col("a.scad_primary_flag"),
            F.col("a.scad_quality_code"),
            F.col("a.scad_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )

    # -------------------- 2. tadobecampaign：取adobe中最新 opt-in 状态（只取acs source） --------------------
    campaign_df = (consumer_acs_df.alias("c")
        .join(address_df.alias("a"), 
            (F.col("c.scon_id") == F.col("a.scad_scon_id")) & 
            (F.col("c.scon_mrkt_code") == F.col("a.scad_mrkt_code")),
            "inner"
        )
        .join(filter_optin_df.alias("o"),
            (F.col("a.scad_scon_id") == F.col("o.scop_scon_id")) & 
            (F.col("a.scad_mrkt_code") == F.col("o.scop_mrkt_code")) 
            # & (F.col("o.scop_comm_code") == comm_code_expr)
            ,
            "inner"
        )
        .select(
            # *group_cols,
            # F.col("c.scon_id"), F.col("c.scon_update_dt"), F.col("c.scon_srcs_code"), F.col("c.scon_delete_flag"),
            F.col("scon_mrkt_code"), 
            F.col("scon_brnd_code"),
            # F.col("a.scad_id"),
            # F.col("a.scad_addt_code"),
            # F.col("a.scad_sourcetimestamp"),
            F.col("a.scad_address1"),
            F.col("a.scad_address2"),
            F.col("a.scad_address3"),
            F.col("a.scad_city_localdesc"),
            F.col("a.scad_prvn_localdesc"),
            F.col("a.scad_cntr_isoalpha3code"),
            F.col("a.scad_postalcode"),
            F.col("a.scad_validitycode"),
            F.col("a.scad_primary_flag"),
            F.col("a.scad_quality_code"),
            F.col("a.scad_quality_desc"),
            F.col("o.scop_comm_code").alias("comm_code"),
            F.col("o.scop_optin_dt").alias("optin_date"),
            F.col("o.scop_optin_flag").alias("optin_flag")
        )
        # .filter(F.col("comm_code") == comm_code_expr) \
        .withColumn("max_optin", F.max("optin_date").over(window_campaign)) \
        .filter(F.col("optin_date") == F.col("max_optin")) \
        .select(
                *[F.col(k).alias(f"vcad_{k}") for k in campaign_keys],
                F.col("scad_quality_code").alias("vcad_add_quality_code"),
                F.col("scad_quality_desc").alias("vcad_add_quality_desc"),
                F.col("comm_code").alias("vcad_add_comm_code"),
                F.col("optin_date").alias("vcad_add_optin_dt"),
                F.col("optin_flag").alias("vcad_add_optin_flag")
        ).dropDuplicates([f"vcad_{k}" for k in campaign_keys])
    )
    
    # -------------------- 3. 核心宽表：基础数据 + Campaign（不取acs source） --------------------
    join_cond = [F.col(f"base.{k}") == F.col(f"camp.vcad_{k}") for k in campaign_keys]
    join_cond.append(F.col("base.comm_code") == F.col("camp.vcad_add_comm_code"))
    # 只取optin表optin_date<= adobe表 optin_dt的数据
    join_cond.append(F.col("base.optin_date") <= F.col("camp.vcad_add_optin_dt"))

    base_campaign_df = base_df.alias("base") \
        .join(campaign_df.alias("camp"), join_cond, "left_outer") \
        .select(
            *[F.col(f"base.{col}").alias(col) for col in group_cols],
            F.col("base.scon_id"),
            F.col("base.scon_update_dt"),
            F.col("base.scon_srcs_code"), F.col("base.scon_delete_flag"),
            F.col("base.scad_id"),
            F.col("base.scad_addt_code"),
            F.col("base.scad_sourcetimestamp"),
            F.col("base.scad_address1"),
            F.col("base.scad_address2"),
            F.col("base.scad_address3"),
            F.col("base.scad_city_localdesc"),
            F.col("base.scad_prvn_localdesc"),
            F.col("base.scad_cntr_isoalpha3code"),
            F.col("base.scad_postalcode"),
            F.col("base.scad_validitycode"),
            F.col("base.scad_primary_flag"),
            F.coalesce(F.col("camp.vcad_add_quality_code"), F.col("base.scad_quality_code")).alias("QualityCode"),
            F.coalesce(F.col("camp.vcad_add_quality_desc"), F.col("base.scad_quality_desc")).alias("QualityDesc"),
            F.coalesce(F.col("camp.vcad_add_comm_code"), F.col("base.comm_code")).alias("OptinCommCode"),
            F.coalesce(F.col("camp.vcad_add_optin_dt"), F.col("base.optin_date")).alias("OptinDate"),
            F.coalesce(F.col("camp.vcad_add_optin_flag"), F.col("base.optin_flag")).alias("OptinFlag")
        )

    # -------------------- 4. 窗口计算（MaxOptInDate、Ovl_MaxOptInDate）--------------------
    # 4.1 每个细粒度分组的MaxOptInDate（带address）
    # 4.2 取细粒度OptinDate=MaxOptInDate的记录
    # 4.2 每个粗粒度分组的Ovl_MaxOptInDate（不带address）
    base_maxoptin_df = base_campaign_df \
        .withColumn("MaxOptInDate",
            F.coalesce(
                # 优先级1：排除指定srcs_code + scon_delete_flag <> 1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), F.col("OptinDate"))).over(win_code_value),
                # 优先级2：仅排除指定srcs_code
                F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE), F.col("OptinDate"))).over(win_code_value),
                # 优先级3：仅scon_delete_flag <> 1
                F.max(F.when(F.col("scon_delete_flag") != 1, F.col("OptinDate"))).over(win_code_value),
                # 默认：全部记录的最大值
                F.max("OptinDate").over(win_code_value)
            )) \
        .filter(F.col("OptinDate") == F.col("MaxOptInDate")) \
        .withColumn("Ovl_MaxOptInDate",
            F.coalesce(
                # 优先级1：排除指定srcs_code + QualityCode=["vld", "und"] + OptinFlag=1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级2：排除指定srcs_code + QualityCode=["vld", "und"]
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & F.col("QualityCode").isin(VALID_QUALITY), F.col("OptinDate"))).over(win_code),
                # 优先级3：排除指定srcs_code + OptinFlag=1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级4：仅排除指定srcs_code
                F.max(F.when(~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE), F.col("OptinDate"))).over(win_code),
                # 优先级5：QualityCode=["vld", "und"] + OptinFlag=1
                F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY) & (F.col("OptinFlag") == 1), F.col("OptinDate"))).over(win_code),
                # 优先级6：仅QualityCode=["vld", "und"]
                F.max(F.when(F.col("QualityCode").isin(VALID_QUALITY), F.col("OptinDate"))).over(win_code),
                # 优先级7：仅OptinFlag=1
                F.max(F.when(F.col("OptinFlag") == 1, F.col("OptinDate"))).over(win_code),
                # 默认：全部记录的最大值
                F.max("OptinDate").over(win_code)
            ))

    # -------------------- 5. 分别计算 有Ovl_MaxOptInDate和无Ovl_MaxOptInDate 的最大时间戳 --------------------
    # 有 ovl 的组：取 OptinDate = Ovl_MaxOptInDate 的记录中的最大时间戳
    max_ts_ovl_df = base_maxoptin_df \
        .filter(F.col("Ovl_MaxOptInDate").isNotNull()) \
        .filter(F.col("OptinDate") == F.col("Ovl_MaxOptInDate")) \
        .groupBy(*group_cols_code) \
        .agg(F.max("scad_sourcetimestamp").alias("maxtimestamp"))

    # 无 ovl 的组：过滤出无ovl的组（Ovl_MaxOptInDate为null），按另一套优先级取最大时间戳
    # max_ts_no_ovl_df = base_maxoptin_df.filter(F.col("Ovl_MaxOptInDate").isNull()) \
    max_ts_no_ovl_df = base_campaign_df.join(max_ts_ovl_df, group_cols_code, "left_anti") \
        .withColumn("maxtimestamp",
            F.coalesce(
                # 优先级1：排除指定srcs_code + scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), F.col("scad_sourcetimestamp"))).over(win_code),
                # 优先级2：排除指定srcs_code + scon_delete_flag <> 1
                F.max(F.when((~F.col("scon_srcs_code").isin(EXCLUDE_SOURCE)) & (F.col("scon_delete_flag") != 1), F.col("scad_sourcetimestamp"))).over(win_code),
                # 优先级3：scon_delete_flag <> 1 + QualityCode=["vld", "und"]
                F.max(F.when((F.col("scon_delete_flag") != 1) & F.col("QualityCode").isin(VALID_QUALITY), F.col("scad_sourcetimestamp"))).over(win_code),
                # 优先级4：仅scon_delete_flag <> 1
                F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scad_sourcetimestamp"))).over(win_code),
                # 默认：全部记录的最大值
                F.max("scad_sourcetimestamp").over(win_code)
            )
        ).select(*group_cols_code, "maxtimestamp").distinct()

    # 合并两部分最大时间戳
    all_max_ts = max_ts_ovl_df.union(max_ts_no_ovl_df)

    # -------------------- 6. 根据最大时间戳获取最大scon_id --------------------
    #  定义窗口：按分组分区，排序规则
    #   - 优先级1：只保留时间戳等于 maxtimestamp 的记录（即筛选出该分组内最新的一批记录）
    #   - 优先级2：在这些最新记录中，先按scon_update_dt降序，再按scon_id降序取第一条
    win_final = Window.partitionBy(*group_cols_code).orderBy(
        F.when(F.col("scad_sourcetimestamp") == F.col("maxtimestamp"), 0).otherwise(1).asc(),
        F.col("scon_update_dt").desc(),
        F.col("scon_id").desc(),
        F.col("scad_id").desc(),
        F.col("OptinCommCode")
    )

    base_with_ts_df = base_campaign_df.join(all_max_ts, group_cols_code, "left") \
        .withColumn("rn", F.row_number().over(win_final)) \
        .filter(F.col("rn") == 1)

    # -------------------- 7. 输出两个df --------------------
    # 后面生成optin json时使用
    bDerivedAddressOptin_df = base_with_ts_df.select(
        *group_cols_code,
        F.col("scon_id").alias("Maxsconid"),
        F.col("OptinCommCode"),
        F.col("OptinDate"),
        F.col("OptinFlag")
       ).distinct()
    
    # 生成address json时使用
    json_df = base_with_ts_df
    # .dropDuplicates(group_cols_code)
    # .orderBy(group_cols_code)

    # 构建address结构体
    address_struct = F.struct(
        F.col("scad_addt_code").alias("@TypeCode"),
        F.date_format(F.col("scad_sourcetimestamp"), timestamp_format).alias("SourceTimestamp"),
        F.col("scad_address1").alias("Address1"),
        F.col("scad_address2").alias("Address2"),
        F.col("scad_address3").alias("Address3"),
        F.lit(None).alias("Address4"),
        F.lit(None).alias("Address5"),
        F.lit(None).alias("FlatNo"),
        F.lit(None).alias("Floor"),
        F.lit(None).alias("Block"),
        F.lit(None).alias("Phase"),
        F.lit(None).alias("Building"),
        F.lit(None).alias("StreetNo"),
        F.lit(None).alias("StreetNoSuffix"),
        F.lit(None).alias("Alley"),
        F.lit(None).alias("StreetName"),
        F.lit(None).alias("StreetName2"),
        F.lit(None).alias("Estate"),
        F.lit(None).alias("LaneNo"),
        F.lit(None).alias("LaneName"),
        F.lit(None).alias("Sector"),
        F.lit(None).alias("POBox"),
        F.col("scad_postalcode").alias("PostalCode"),
        F.lit(None).alias("SubCityCode"),
        F.lit(None).alias("SubCityCodeDescription_en"),
        F.lit(None).alias("SubCityCodeDescription_local"),
        F.lit(None).alias("CityCode"),
        F.lit(None).alias("CityDescription_en"),
        F.col("scad_city_localdesc").alias("CityDescription_local"),
        F.lit(None).alias("Province"),
        F.lit(None).alias("ProvinceCode"),
        F.lit(None).alias("ProvinceDescription_en"),
        F.col("scad_prvn_localdesc").alias("ProvinceDescription_local"),
        F.lit(None).alias("AdminArea"),
        F.col("scad_cntr_isoalpha3code").alias("CountryCode_ISO3"),
        F.lit(None).alias("NUTSCode"),
        F.lit(None).alias("Longitude"),
        F.lit(None).alias("Latitude"),
        F.lit(None).alias("GeocodeResolution"),
        F.lit(None).alias("GeographicalSpokenLanguage"),
        F.col("QualityCode").alias("DataQualityCode"),
        F.col("QualityDesc").alias("DataQualityDescription")
    )
    
    # 生成最终DataFrame
    final_df = (json_df
        .groupBy(*group_cols)
        .agg( F.collect_list(address_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("Address")), options={"ignoreNullFields": "false"}).alias("AddressJSON")
        ))

    return bDerivedAddressOptin_df, final_df

In [0]:
def generate_optin_json(consumer_df, bDerivedEmailOptin_df, bDerivedPhoneOptin_df, bDerivedAddressOptin_df, optin_df, group_cols):
    
    group_cols_code = group_cols + ["scop_comm_code"]
    win_core = Window.partitionBy(*group_cols_code)

    # -------------------- 1. 合并得到 三个ContactOptin --------------------
    contact_optin_df = (bDerivedEmailOptin_df.union(bDerivedPhoneOptin_df).union(bDerivedAddressOptin_df).distinct() \
    .filter(F.col("OptinDate").isNotNull()) \
    .select(
        *group_cols,
        F.col("Maxsconid").alias("scop_id"),
        F.col("OptinDate").alias("scop_optin_dt"),
        F.col("OptinCommCode").alias("scop_comm_code"),
        F.col("OptinFlag").alias("scop_optin_flag")
    )
    .withColumn("rn", F.row_number().over(win_core.orderBy(F.col("scop_optin_dt").desc(),F.col("scop_optin_flag").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
    )
    
    # -------------------- 2. 处理未出现在 contact_optin 中的其他 optin 记录 --------------------
    existing_combinations_df = contact_optin_df.select(*group_cols_code).distinct()

    # 计算每个组合的最大optin_dt（优先取未删除记录）,筛选出optin_dt等于最大值的记录
    # 计算每个组合的最大consumerid, 筛选consumerid等于最大值的记录
    # JPN 特有过滤 "eml", "mbleml"
    other_optin_df = consumer_df.alias("c") \
        .join(optin_df.alias("o"), 
            (F.col("o.scop_scon_id") == F.col("c.scon_id")) & 
            (F.col("o.scop_mrkt_code") == F.col("c.scon_mrkt_code")),
            "inner"
        ) \
        .join(existing_combinations_df.alias("e"), group_cols_code, "left_anti") \
        .filter(~((F.col("c.scon_mrkt_code") == "JPN") & F.col("o.scop_comm_code").isin(["eml", "mbleml"]))) \
        .withColumn("max_optin_dt",
                    F.coalesce(
                        F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scop_optin_dt"))).over(win_core),
                        F.max("scop_optin_dt").over(win_core)
                   )) \
        .filter(F.col("scop_optin_dt") == F.col("max_optin_dt")) \
        .withColumn("rn", F.row_number().over(win_core.orderBy(F.col("scon_consumerid").desc(), F.col("scop_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn") \
        .select(
            *group_cols,
            F.col("scop_id"),
            F.col("scop_optin_dt"),
            F.col("scop_comm_code"),
            F.col("scop_optin_flag")
        )
    
    # -------------------- 3. 合并两部分，按分组键去重 --------------------
    base_df = contact_optin_df.union(other_optin_df) \
        .dropDuplicates(group_cols_code)
        # .orderBy(*group_cols_code)

    # 构建optin结构体
    optin_struct = F.struct(
        F.date_format(F.col("scop_optin_dt"), timestamp_format).alias("OptInTimestamp"),
        F.col("scop_comm_code").alias("CommunicationChannelCode"),
        F.col("scop_optin_flag").alias("OptInFlag"),
        F.lit(None).alias("OptInSourceSystemCode")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(optin_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("OptIn")), options={"ignoreNullFields": "false"}).alias("OptInJSON")
        ))

    return final_df

In [0]:
def generate_crossbrand_optin_json(consumer_df, crossbrand_optin_table, group_cols):

    win_max_dt = Window.partitionBy(*group_cols)
    # win_max_consumerid = Window.partitionBy(*group_cols + ["scbo_optin_dt"])

    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id", "scon_consumerid", "scon_delete_flag")

    # 读crossbrand_optin表
    crossbrand_optin_df = spark.table(crossbrand_optin_table) \
        .select("scbo_scon_id", "scbo_id", "scbo_mrkt_code", "scbo_optin_dt", "scbo_optin_flag")

    cond = ((sconsumer_df.scon_id == crossbrand_optin_df.scbo_scon_id) &
            (sconsumer_df.scon_mrkt_code == crossbrand_optin_df.scbo_mrkt_code)
           )

    # 关联两张表，计算max_optin_dt和max_consumerid
    # 筛选：当前记录optin_dt = 分组最大optin_dt
    # 筛选：当前记录的consumerid = 分组最大consumerid
    base_df = sconsumer_df.join(crossbrand_optin_df, cond, "inner") \
        .withColumn("max_optin_dt",
            F.coalesce(
                F.max(F.when(F.col("scon_delete_flag") != 1, F.col("scbo_optin_dt"))).over(win_max_dt),
                F.max(F.col("scbo_optin_dt")).over(win_max_dt))) \
        .filter(F.col("scbo_optin_dt") == F.col("max_optin_dt")) \
        .withColumn("rn", F.row_number().over(win_max_dt.orderBy(F.col("scon_consumerid").desc(), F.col("scbo_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn") \
        .dropDuplicates(group_cols) \
        # .orderBy(group_cols)

    # 构建crossbrand_optin结构体
    crossbrand_optin_struct = F.struct(
        F.date_format(F.col("scbo_optin_dt"), timestamp_format).alias("OptInTimestamp"),
        F.col("scbo_optin_flag").alias("OptInFlag")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(crossbrand_optin_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("CrossBrandOptIn")), options={"ignoreNullFields": "false"}).alias("CrossBrandOptInJSON")
        ))

    return final_df

In [0]:
def generate_attributes_json(consumer_df, attributes_table, group_cols):

    win_special_max = Window.partitionBy(*group_cols + ["a.sccu_name"])
    win_reg_dt_count = Window.partitionBy(*group_cols)
    
    # 抽离ATTR_NAMES常量（便于维护）
    SPECIAL_ATTR_NAMES = ['Derived_Location', 'Derived_City', 'Derived_Province', 'Derived_South_China_Flag', 'South_China_Phone_Flag']

    # 读consumer表
    sconsumer_df = consumer_df

    # 读attributes表
    attributes_df = spark.table(attributes_table).select("sccu_scon_id", "sccu_mrkt_code", "sccu_name", "sccu_value")

    # 关联两张表
	# 计算：special_names的max_value
	# 计算：reg_dt去重计数
	# 计算：DTCFlag分支的条件标记
    base_df = sconsumer_df.alias("c") \
        .join(attributes_df.alias("a"), (F.col("c.scon_id") == F.col("a.sccu_scon_id")) & (F.col("c.scon_mrkt_code") == F.col("a.sccu_mrkt_code")), "left") \
        .withColumn("special_max_value", F.when(F.col("a.sccu_name").isin(SPECIAL_ATTR_NAMES), F.max("a.sccu_value").over(win_special_max) )) \
        .withColumn("reg_dt_count", F.approx_count_distinct("c.scon_reg_dt").over(win_reg_dt_count)) \
        .withColumn("is_dtc_flag",
            F.when(
                (F.coalesce(F.col("c.tcpm_touchpointtypecode"), F.lit("")) == "frestnstr") |
                ((F.coalesce(F.col("c.tcpm_distributionchannelcode"), F.lit("")) == "ecm") & F.col("c.scon_registration_toch_code").like("EC%")),
                F.lit(True)
            ).otherwise(F.lit(False)) )

    # 生成所有分支的键值对（替代4个UNION）
    final_raw_df = base_df \
        .withColumn(
            "attr_array",
            F.array(
                # 分支1：非特殊属性
                F.when(
                    ~F.col("a.sccu_name").isin(SPECIAL_ATTR_NAMES) & F.col("a.sccu_name").isNotNull(),
                    F.struct(F.col("a.sccu_name").alias("name"), F.col("a.sccu_value").alias("value"))
                ),
                # 分支2：特殊属性
                F.when(
                    F.col("a.sccu_name").isin(SPECIAL_ATTR_NAMES) & F.col("a.sccu_name").isNotNull(),
                    F.struct(F.col("a.sccu_name").alias("name"), F.col("special_max_value").alias("value"))
                ),
                # 分支3：DTCFlag
                F.when(
                    F.col("is_dtc_flag") == True,
                    F.struct(F.lit("DTCFlag_ByRegistration").alias("name"), F.lit(1).alias("value"))
                ),
                # 分支4：RegDate
                F.when(
                    F.col("reg_dt_count") > 1,
                    F.struct(F.lit("RegDate").alias("name"), F.col("c.scon_reg_dt").alias("value"))
                ) )) \
        .withColumn("attr", F.explode("attr_array")) \
        .filter(F.col("attr").isNotNull()) \
        .select(
            *group_cols,
            F.col("attr.name").alias("sccu_name"),
            F.col("attr.value").alias("sccu_value")) \
        .distinct() \
        # .orderBy(*group_cols, "sccu_name", "sccu_value")

    # 构建CustomAttributes结构体
    attributes_struct = F.struct(
        F.col("sccu_name").alias("@Name"),
        F.col("sccu_value").alias("@Value")
    )

    # 生成最终DataFrame
    final_df = (final_raw_df
        .groupBy(*group_cols)
        .agg( F.collect_list(attributes_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("CustomAttribute")), options={"ignoreNullFields": "false"}).alias("CustomAttributesJSON")
        ))

    return final_df

In [0]:
def generate_auxiliary_json(consumer_df, auxiliary_table, group_cols):
    # 抽离需要排除的scaa_desc常量（便于维护）
    EXCLUDE_SCAA_DESC = ['BLUE_OCEAN_ANSWERS','CLINICAL_REALITY_ANSWERS','FOUNDATION_FINDER_ANSWERS']

    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读auxiliary表
    auxiliary_df = spark.table(auxiliary_table) \
        .select("scaa_scon_id", "scaa_mrkt_code", "scaa_code", "scaa_desc", "scaa_multivalueflag", "scaa_value", "scaa_active_flag")

    cond = ((sconsumer_df.scon_id == auxiliary_df.scaa_scon_id) &
            (sconsumer_df.scon_mrkt_code == auxiliary_df.scaa_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(auxiliary_df, cond, "inner") \
        .filter(~F.col("scaa_desc").isin(EXCLUDE_SCAA_DESC)) \
        .dropDuplicates(group_cols + ["scaa_code", "scaa_desc", "scaa_value"]) \
        # .orderBy(*group_cols + ["scaa_code", "scaa_desc", "scaa_value"])

    # 构建auxiliary结构体
    auxiliary_struct = F.struct(
        F.col("scaa_code").alias("Code"),
        F.col("scaa_desc").alias("Description"),
        F.col("scaa_multivalueflag").alias("MultiValue"),
        F.col("scaa_value").alias("Value"),
        F.col("scaa_active_flag").alias("Active")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(auxiliary_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("AuxiliaryAttribute")), options={"ignoreNullFields": "false"}).alias("AuxiliaryAttributesJSON")
        ))

    return final_df

In [0]:
def generate_hair_type_json(consumer_df, hair_type_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读hair_type表
    hair_type_df = spark.table(hair_type_table).select("scht_scon_id", "scht_mrkt_code", "scht_hairtype")

    cond = ((sconsumer_df.scon_id == hair_type_df.scht_scon_id) &
            (sconsumer_df.scon_mrkt_code == hair_type_df.scht_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(hair_type_df, cond, "inner") \
        .select(*group_cols, "scht_hairtype").distinct() \
        # .orderBy(*group_cols, "scht_hairtype")

    # 构建hair_type结构体
    hair_type_struct = F.struct(
        F.col("scht_hairtype").alias("HairType")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(F.col("scht_hairtype")).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("HairType")), options={"ignoreNullFields": "false"}).alias("HairTypeJSON")
        ))

    return final_df

In [0]:
def generate_makeup_concerns_json(consumer_df, makeup_concerns_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读makeup_concerns表
    makeup_concerns_df = spark.table(makeup_concerns_table).select("scmc_scon_id", "scmc_mrkt_code", "scmc_makeupconcern_desc")

    cond = ((sconsumer_df.scon_id == makeup_concerns_df.scmc_scon_id) &
            (sconsumer_df.scon_mrkt_code == makeup_concerns_df.scmc_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(makeup_concerns_df, cond, "inner") \
        .select(*group_cols, "scmc_makeupconcern_desc").distinct() \
        # .orderBy(*group_cols, "scmc_makeupconcern_desc")

    # 构建makeup_concerns结构体
    makeup_concerns_struct = F.struct(
        F.col("scmc_makeupconcern_desc").alias("MakeUpConcerns")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(F.col("scmc_makeupconcern_desc")).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("MakeUpConcerns")), options={"ignoreNullFields": "false"}).alias("MakeUpConcernsJSON")
        ))

    return final_df

In [0]:
def generate_hair_concerns_json(consumer_df, hair_concerns_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读hair_concerns表
    hair_concerns_df = spark.table(hair_concerns_table).select("schc_scon_id", "schc_mrkt_code", "schc_hairconcern_desc")

    cond = ((sconsumer_df.scon_id == hair_concerns_df.schc_scon_id) &
            (sconsumer_df.scon_mrkt_code == hair_concerns_df.schc_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(hair_concerns_df, cond, "inner") \
        .select(*group_cols, "schc_hairconcern_desc").distinct() \
        # .orderBy(*group_cols, "schc_hairconcern_desc")

    # 构建hair_concerns结构体
    hair_concerns_struct = F.struct(
        F.col("schc_hairconcern_desc").alias("HairConcerns") 
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(F.col("schc_hairconcern_desc")).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("HairConcerns")), options={"ignoreNullFields": "false"}).alias("HairConcernsJSON")
        ))

    return final_df

In [0]:
def generate_skin_concerns_json(consumer_df, skin_concerns_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读skin_concerns表
    skin_concerns_df = spark.table(skin_concerns_table).select("scsc_scon_id", "scsc_mrkt_code", "scsc_skinconcern_desc")

    cond = ((sconsumer_df.scon_id == skin_concerns_df.scsc_scon_id) &
            (sconsumer_df.scon_mrkt_code == skin_concerns_df.scsc_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(skin_concerns_df, cond, "inner") \
        .select(*group_cols, "scsc_skinconcern_desc").distinct() \
        # .orderBy(*group_cols, "scsc_skinconcern_desc")

    # 构建skin_concerns结构体
    skin_concerns_struct = F.struct(
        F.col("scsc_skinconcern_desc").alias("SkinConcerns") 
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(F.col("scsc_skinconcern_desc")).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("SkinConcerns")), options={"ignoreNullFields": "false"}).alias("SkinConcernsJSON")
        ))

    return final_df

In [0]:
def generate_terms_json(consumer_df, terms_table, group_cols):

    win_max_version = Window.partitionBy(*group_cols + ["scte_terms_code"])
    win_max_accept_dt = Window.partitionBy(*group_cols + ["scte_terms_code", "scte_terms_version"])

    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id", "scon_consumerid", "scon_delete_flag")

    # 读terms表
    terms_df = spark.table(terms_table) \
        .select("scte_scon_id", "scte_mrkt_code", "scte_terms_code", "scte_terms_description", "scte_terms_accept_dt", "scte_terms_version")

    cond = ((sconsumer_df.scon_id == terms_df.scte_scon_id) &
            (sconsumer_df.scon_mrkt_code == terms_df.scte_mrkt_code)
           )

    # 关联两张表
    # 筛选出仅最大版本号的记录（剔除历史低版本）
    # 筛选出仅最大接受日期的记录（取最新接受的条款）
    base_df = sconsumer_df.join(terms_df, cond, "inner") \
        .withColumn("max_version", F.max(F.col("scte_terms_version")).over(win_max_version)) \
        .withColumn("max_accept_dt", F.max(F.col("scte_terms_accept_dt")).over(win_max_accept_dt)) \
        .filter(F.col("scte_terms_version") == F.col("max_version")) \
        .filter(F.col("scte_terms_accept_dt") == F.col("max_accept_dt")) \
        .dropDuplicates(group_cols + ["scte_terms_code"]) \
        # .orderBy(group_cols + ["scte_terms_code"])

    # 构建terms结构体
    terms_struct = F.struct(
        F.col("scte_terms_code").alias("Code"),
        F.col("scte_terms_description").alias("Description"),
        F.col("scte_terms_version").alias("Version"),
        F.date_format(F.col("scte_terms_accept_dt"), date_format).alias("AcceptedDate")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(terms_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("TermsAndCondition")), options={"ignoreNullFields": "false"}).alias("TermsJSON")
        ))

    return final_df

In [0]:
def generate_consumer_group_json(consumer_df, consumer_group_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读consumer_group表
    consumer_group_df = spark.table(consumer_group_table).select("scgr_scon_id", "scgr_mrkt_code", "scgr_consumer_grp")

    cond = ((sconsumer_df.scon_id == consumer_group_df.scgr_scon_id) &
            (sconsumer_df.scon_mrkt_code == consumer_group_df.scgr_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(consumer_group_df, cond, "inner") \
        .select(*group_cols, "scgr_consumer_grp").distinct() \
        # .orderBy(*group_cols, "scgr_consumer_grp")

    # 构建consumer_group结构体
    consumer_group_struct = F.struct(
        F.col("scgr_consumer_grp").alias("ConsumerGroup") 
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(F.col("scgr_consumer_grp")).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("CustomerGroup")), options={"ignoreNullFields": "false"}).alias("ConsumerGroupJSON")
        ))

    return final_df

In [0]:
def generate_remark_json(consumer_df, remark_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读remark表
    remark_df = spark.table(remark_table).select( "scre_scon_id", "scre_mrkt_code", "scre_rmak_dt", "scre_rmak_code", "scre_rmak_description")

    cond = ((sconsumer_df.scon_id == remark_df.scre_scon_id) &
            (sconsumer_df.scon_mrkt_code == remark_df.scre_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(remark_df, cond, "inner") \
        .select(*group_cols, "scre_rmak_dt", "scre_rmak_code", "scre_rmak_description").distinct() \
        # .orderBy(*group_cols, "scre_rmak_dt", "scre_rmak_code", "scre_rmak_description")

    # 构建remark结构体
    remark_struct = F.struct(
        F.col("scre_rmak_code").alias("RemarkCode"),
        F.date_format(F.col("scre_rmak_dt"), date_format).alias("RemarksDate"),
        F.col("scre_rmak_description").alias("Remarks")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(remark_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("Remark")), options={"ignoreNullFields": "false"}).alias("RemarkJSON")
        ))

    return final_df

In [0]:
def generate_notes_json(consumer_df, notes_table, group_cols):
    # 读consumer表
    sconsumer_df = consumer_df \
        .select(*group_cols, "scon_id")

    # 读notes表
    notes_df = spark.table(notes_table).select("scno_scon_id", "scno_mrkt_code", "scno_seq_num", "scno_type_code", "scno_location", "scno_note")

    cond = ((sconsumer_df.scon_id == notes_df.scno_scon_id) &
            (sconsumer_df.scon_mrkt_code == notes_df.scno_mrkt_code)
           )

    # 关联两张表
    base_df = sconsumer_df.join(notes_df, cond, "inner") \
        .select(*group_cols, "scno_seq_num", "scno_type_code", "scno_location", "scno_note").distinct() \
        # .orderBy(*group_cols, "scno_seq_num", "scno_type_code", "scno_location", "scno_note")

    # 构建notes结构体
    notes_struct = F.struct(
        F.col("scno_seq_num").alias("SeqNum"),
        F.col("scno_type_code").alias("Type"),
        F.col("scno_location").alias("Location"),
        F.col("scno_note").alias("Note"),
        F.lit(None).alias("CreateDate"),
        F.lit(None).alias("CreateBy"),
        F.lit(None).alias("UpdateDate"),
        F.lit(None).alias("UpdateBy")
    )
    
    # 生成最终DataFrame
    final_df = (base_df
        .groupBy(*group_cols)
        .agg( F.collect_list(notes_struct).alias("item_list"))
        .select(
            *group_cols,
            F.to_json(F.struct(F.col("item_list").alias("Note")), options={"ignoreNullFields": "false"}).alias("NotesJSON")
        ))

    return final_df

In [0]:
def fetch_cbr_records(consumer_df, overall_maxtimestamp_df, group_cols):

    window_core = Window.partitionBy(*group_cols)

    # 1. 基础数据读取
    base_df = consumer_df.join(overall_maxtimestamp_df, group_cols, "inner") \
     .select(
        *group_cols, 
        "scon_id", "scon_aff_code", "scon_dvsn_code", "scon_srcs_code", "scon_sourcetimestamp",
        "scon_reg_dt", "scon_firstpurchasedate", "scon_salutation",
        "scon_englishname_quality_code", "scon_localname_quality_code", "scon_localname2_quality_code",
        "scon_englishfirstname", "scon_englishmiddlename", "scon_englishlastname", "scon_englishfullname",
        "scon_localfirstname", "scon_localmiddlename", "scon_locallastname", "scon_localfullname",
        "scon_localfirstname2", "scon_localmiddlename2", "scon_locallastname2", "scon_localfullname2",
        "scon_gndr_code", "scon_birthday", "scon_birthmonth", "scon_birthyear",
        "scon_identitynum", "scon_passportnum", "scon_socialsecuritynum", "scon_clas_code",
        "scon_registration_toch_code", "scon_preferred_toch_code", "scon_assigned_prsn_code",
        "scon_registration_prsn_code", "scon_wlng_code", "scon_slng_code", "scon_cntr_isoalpha3code",
        "scon_ethn_code", "scon_sknt_code", "scon_hairt_code", "scon_cvls_code", "scon_company",
        "scon_department", "scon_jobtitle", "scon_yearlyincome", "scon_donotcontact_flag", "scon_curr_code",
        "scon_agefrom", "scon_ageto", "scon_nationality", "scon_channel", "scon_preferred_comm_channel",
        "scon_anniversary_dt", "scon_commercial_flag", "scon_emailreceipt_flag", "scon_prospect_flag",
        "scon_active_flag", "scon_hrrequesttimestamp", "scon_status", "scon_delete_flag", "scon_consumerid", "maxoveralltimestamp_contact","scon_update_dt"
    )

    # 2. 窗口计算
    # is_latest_source：max scon_sourcetimestamp
    # is_earliest_reg：min scon_reg_dt
    # is_latest_birth：max scon_sourcetimestamp by birth info
    base_df = add_maxsourcetimestamp_by_market(base_df, group_cols) \
        .withColumn("consumerclasscode", F.max(F.when(F.coalesce(F.col("scon_clas_code"), F.lit("")) == "nnrgl", "nnrgl")).over(window_core)) \
        .withColumn("maxbirthtimestamp", 
                F.coalesce(
                    # 第一优先级：有birthyear
                    F.max(F.when(
                        F.coalesce(F.col("scon_birthyear"), F.lit("")) != "",
                        F.col("scon_sourcetimestamp")
                    )).over(window_core),
                    # 第二优先级：有birthcombo
                    F.max(F.when(
                        F.concat_ws("", "scon_birthday", "scon_birthmonth", "scon_birthyear") != "",
                        F.col("scon_sourcetimestamp")
                    )).over(window_core),
                    # 默认
                    F.max("scon_sourcetimestamp").over(window_core)
                )) \
        .withColumn("maxagerangetimestamp",
                    F.coalesce(F.max(F.when(F.concat_ws("", F.col("scon_agefrom"), F.col("scon_ageto")) != "",F.col("scon_sourcetimestamp"))).over(window_core),
                               F.max("scon_sourcetimestamp").over(window_core)
                               )) \
        .withColumn("maxgendercodetimestamp",
                    F.coalesce(F.max(F.when(F.coalesce(F.col("scon_gndr_code"), F.lit("")) != "", F.col("scon_sourcetimestamp"))).over(window_core),
                               F.max("scon_sourcetimestamp").over(window_core)
                               )) \
        .withColumn("dense_rank_cid", F.dense_rank().over(window_core.orderBy("scon_consumerid"))) \
        .withColumn("distinct_consumerid_count", F.max("dense_rank_cid").over(window_core)) \
        .withColumn("has_onlshell", F.max(F.when(F.coalesce(F.col("scon_clas_code"), F.lit("")) == "onlshell", 1).otherwise(0)).over(window_core) > 0) \
        .withColumn("minregdate",
            F.when(
            # TWN特殊逻辑：count(distinct scon_consumerid)>1 and scon_clas_code=onlshell取max scon_reg_dt，其他情况取min scon_reg_dt
            (F.col("scon_mrkt_code") == "TWN") & (F.col("distinct_consumerid_count") > 1) & F.col("has_onlshell"), 
            F.max("scon_reg_dt").over(window_core)
            ).otherwise(F.min("scon_reg_dt").over(window_core)) ) \
        .withColumn("min_firstpurchasedate", F.min("scon_firstpurchasedate").over(window_core)) \
        .withColumn("is_latest_source", F.col("scon_sourcetimestamp") == F.col("maxsourcetimestamp")) \
        .withColumn("is_earliest_reg", F.col("scon_reg_dt") == F.col("minregdate")) \
        .withColumn("is_latest_birth", F.col("scon_sourcetimestamp") == F.col("maxbirthtimestamp")) \
        .withColumn("is_latest_agerange", F.col("scon_sourcetimestamp") == F.col("maxagerangetimestamp")) \
        .withColumn("is_latest_gender", F.col("scon_sourcetimestamp") == F.col("maxgendercodetimestamp")) \
        .drop("dense_rank_cid", "distinct_consumerid_count")

    # 3.1 最新sourcetimestamp记录并添加NameFilledFlag字段（别名a）
    latest_overall_df = base_df.filter(F.col("is_latest_source"))
    latest_overall_addflag_df = add_namefilledflag_l2l3(latest_overall_df) \
        .withColumn("LastUpdateTimeStamp", F.col("scon_sourcetimestamp")) \
        .withColumn("LastUpdateTouchPointCode", F.col("scon_registration_toch_code")) \
        .withColumn("scon_pre_toch_sourcetimestamp", F.when(F.coalesce(F.col("scon_preferred_toch_code"), F.lit("")) != "", F.col("scon_sourcetimestamp"))) \
        .withColumn("scon_pre_toch_aff_code", F.when(F.coalesce(F.col("scon_preferred_toch_code"), F.lit("")) != "", F.col("scon_aff_code"))) \
        .withColumn("scon_pre_toch_mrkt_code", F.when(F.coalesce(F.col("scon_preferred_toch_code"), F.lit("")) != "", F.col("scon_mrkt_code"))) \
        .withColumn("scon_pre_toch_dvsn_code", F.when(F.coalesce(F.col("scon_preferred_toch_code"), F.lit("")) != "", F.col("scon_dvsn_code"))) \
        .withColumn("scon_pre_toch_brnd_code", F.when(F.coalesce(F.col("scon_preferred_toch_code"), F.lit("")) != "", F.col("scon_brnd_code"))) \
        .withColumn("scon_assn_prsn_srcs_code", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_srcs_code"))) \
        .withColumn("scon_assn_prsn_sourcetimestamp", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_sourcetimestamp"))) \
        .withColumn("scon_assn_prsn_aff_code", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_aff_code"))) \
        .withColumn("scon_assn_prsn_mrkt_code", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_mrkt_code"))) \
        .withColumn("scon_assn_prsn_dvsn_code", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_dvsn_code"))) \
        .withColumn("scon_assn_prsn_brnd_code", F.when(F.coalesce(F.col("scon_assigned_prsn_code"), F.lit("")) != "", F.col("scon_brnd_code"))) \
        .withColumn("scon_ethn_mrkt_code", F.when(F.coalesce(F.col("scon_ethn_code"), F.lit("")) != "", F.col("scon_mrkt_code"))) \
        .withColumn("scon_sknt_brnd_code", F.when(F.coalesce(F.col("scon_sknt_code"), F.lit("")) != "", F.col("scon_brnd_code"))) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.coalesce(F.col("scon_clas_code"), F.lit("")).asc(),F.col("scon_update_dt").desc(),F.col("scon_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn") \
        .withColumn("scon_clas_code", F.when(F.col("consumerclasscode") != "", F.col("consumerclasscode")).otherwise(F.col("scon_clas_code"))) 
        # 原始scon_clas_code需要用于排序,因此最后进行重计算

    # 3.2. 最早reg_dt记录（别名b）
    earliest_reg_df = base_df.filter(F.col("is_earliest_reg")) \
        .withColumn("scon_prsn_srcs_code", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_srcs_code"))) \
        .withColumn("scon_prsn_sourcetimestamp", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_sourcetimestamp"))) \
        .withColumn("scon_prsn_aff_code", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_aff_code"))) \
        .withColumn("scon_prsn_mrkt_code", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_mrkt_code"))) \
        .withColumn("scon_prsn_dvsn_code", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_dvsn_code"))) \
        .withColumn("scon_prsn_brnd_code", F.when(F.coalesce(F.col("scon_registration_prsn_code"), F.lit("")) != "", F.col("scon_brnd_code"))) \
        .select(
            *group_cols,
            "scon_reg_dt", "scon_sourcetimestamp", "scon_registration_toch_code",
            "scon_prsn_srcs_code", "scon_prsn_sourcetimestamp", "scon_prsn_aff_code",
            "scon_prsn_mrkt_code", "scon_prsn_dvsn_code", "scon_prsn_brnd_code",
            "scon_registration_prsn_code", "min_firstpurchasedate",
            "scon_update_dt", "scon_id"
        ) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_sourcetimestamp"), F.col("scon_update_dt"), F.col("scon_id")))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # 3.3 最新生日记录（别名d）
    latest_birth_df = base_df.filter(F.col("is_latest_birth")) \
        .select(
            *group_cols,
            F.col("scon_birthday"),
            F.col("scon_birthmonth"),
            F.col("scon_birthyear"),
            F.col("scon_update_dt"),
            F.col("scon_id")
        ) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_birthyear").desc(), F.col("scon_birthmonth").desc(), F.col("scon_update_dt").desc(), F.col("scon_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")
    
    # 3.4 最新年龄记录（别名 e）
    latest_agerange_df = base_df.filter(F.col("is_latest_agerange")) \
    .select(
        *group_cols,
        F.col("scon_agefrom"),
        F.col("scon_ageto"),
        F.col("scon_update_dt"),
        F.col("scon_id")
    ) \
    .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_update_dt").desc(),F.col("scon_id").desc()))) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

    # 3.5 最新性别记录（别名 f）
    latest_gender_df = base_df.filter(F.col("is_latest_gender")) \
        .select(
            *group_cols,
            F.col("scon_gndr_code"),
            F.col("scon_update_dt"),
            F.col("scon_id")
        ) \
        .withColumn("rn", F.row_number().over(window_core.orderBy(F.col("scon_update_dt").desc(),F.col("scon_id").desc()))) \
        .filter(F.col("rn") == 1) \
        .drop("rn")

    # 4. 最终关联（jpn特殊逻辑：scon_gndr_code字段从scon_gndr_code不为空最大记录中取）
    final_df = latest_overall_addflag_df.alias("a") \
        .join(earliest_reg_df.alias("b"), group_cols, "inner") \
        .join(latest_birth_df.alias("d"), group_cols, "inner") \
        .join(latest_agerange_df.alias("e"), group_cols, "inner") \
        .join(latest_gender_df.alias("f"), group_cols, "inner") \
        .select(
            *group_cols,
            F.col("a.scon_srcs_code"),
            F.col("a.maxoveralltimestamp_contact").alias("scon_sourcetimestamp"),
            F.col("a.scon_aff_code"),
            F.col("a.scon_dvsn_code"),
            F.col("a.scon_salutation"),
            F.col("a.scon_englishfirstname"),
            F.col("a.scon_englishmiddlename"),
            F.col("a.scon_englishlastname"),
            F.col("a.scon_englishfullname"),
            F.col("a.scon_localfirstname"),
            F.col("a.scon_localmiddlename"),
            F.col("a.scon_locallastname"),
            F.col("a.scon_localfullname"),
            F.col("a.scon_localfirstname2"),
            F.col("a.scon_localmiddlename2"),
            F.col("a.scon_locallastname2"),
            F.col("a.scon_localfullname2"),
            F.when(F.col("a.scon_mrkt_code") == "JPN", F.col("f.scon_gndr_code")).otherwise(F.col("a.scon_gndr_code")).alias("scon_gndr_code"),
            F.col("d.scon_birthday"),
            F.col("d.scon_birthmonth"),
            F.col("d.scon_birthyear"),
            F.col("a.scon_identitynum"),
            F.col("a.scon_passportnum"),
            F.col("a.scon_socialsecuritynum"),
            F.col("a.scon_clas_code"),
            F.col("b.scon_reg_dt"),
            F.col("b.scon_sourcetimestamp").alias("scon_reg_sourcetimestamp"),
            F.col("b.scon_registration_toch_code"),
            F.col("b.scon_prsn_srcs_code"),
            F.col("b.scon_prsn_sourcetimestamp"),
            F.col("b.scon_prsn_aff_code"),
            F.col("b.scon_prsn_mrkt_code"),
            F.col("b.scon_prsn_dvsn_code"),
            F.col("b.scon_prsn_brnd_code"),
            F.col("b.scon_registration_prsn_code"),
            F.col("a.LastUpdateTimeStamp").alias("scon_last_update_timestamp"),
            F.col("a.LastUpdateTouchPointCode").alias("scon_last_update_toch_code"),
            F.col("a.scon_pre_toch_sourcetimestamp"),
            F.col("a.scon_pre_toch_aff_code"),
            F.col("a.scon_pre_toch_mrkt_code"),
            F.col("a.scon_pre_toch_dvsn_code"),
            F.col("a.scon_pre_toch_brnd_code"),
            F.col("a.scon_preferred_toch_code"),
            F.col("a.scon_assn_prsn_srcs_code"),
            F.col("a.scon_assn_prsn_sourcetimestamp"),
            F.col("a.scon_assn_prsn_aff_code"),
            F.col("a.scon_assn_prsn_mrkt_code"),
            F.col("a.scon_assn_prsn_dvsn_code"),
            F.col("a.scon_assn_prsn_brnd_code"),
            F.col("a.scon_assigned_prsn_code"),
            F.col("a.scon_wlng_code"),
            F.col("a.scon_slng_code"),
            F.col("a.scon_cntr_isoalpha3code"),
            F.col("a.scon_ethn_mrkt_code"),
            F.col("a.scon_ethn_code"),
            F.col("a.scon_sknt_brnd_code"),
            F.col("a.scon_sknt_code"),
            F.col("a.scon_hairt_code"),
            F.col("a.scon_cvls_code"),
            F.col("a.scon_company"),
            F.col("a.scon_department"),
            F.col("a.scon_jobtitle"),
            F.col("a.scon_yearlyincome"),
            F.col("a.scon_donotcontact_flag"),
            F.col("a.scon_curr_code"),
            F.col("e.scon_agefrom"),
            F.col("e.scon_ageto"),
            F.col("a.scon_nationality"),
            F.col("a.scon_channel"),
            F.col("a.scon_preferred_comm_channel"),
            F.col("a.scon_anniversary_dt"),
            F.col("a.scon_commercial_flag"),
            F.col("a.scon_emailreceipt_flag"),
            F.col("a.scon_prospect_flag"),
            F.col("a.scon_active_flag"),
            F.col("a.scon_hrrequesttimestamp"),
            F.col("a.scon_status"),
            F.col("a.NameFilledFlag"),
            F.lit(None).alias("DervivedBestRecordListID"),
            F.col("b.min_firstpurchasedate").alias("scon_firstpurchasedate")
    ) \

    return final_df

In [0]:
def get_high_count_ukey(cid_limit_num, cid_limit_markets):
    high_count_ukey_df = (spark.table(consumer_table)
        .filter(F.col("scon_mrkt_code").isin(cid_limit_markets))
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .count()
        .filter(F.col("count") > F.lit(cid_limit_num))
    )
    
    return high_count_ukey_df

In [0]:
def get_consumer_data():

    exclude_sources_df = spark.table(dim_excludesource_table)
    exclude_ukey_df = (spark.table(survive_exclude_ukey_table)
        .select(
            F.col("marketcode").alias("exclude_marketcode"),
            F.col("consumermdmkey").alias("exclude_consumermdmkey")
        )
        .distinct()
    )

    # high count ukey  (scon_mrkt_code + consumermdmkey), 需要排除cid过多的ukey
    high_count_ukey_df = get_high_count_ukey(cid_limit_num, cid_limit_markets)
    
    # 关联touchpoint
    touchpoint_master_df = (spark.table(touchpoint_table)
        .withColumn("rn", F.row_number().over(Window.partitionBy("TCPM_MarketCode", "TCPM_BrandCode", "TCPM_SourceSystemCode", "TCPM_TouchPointCode")
            .orderBy(
                F.col("TCPM_SourceTimestamp").desc(),
                F.col("KAFKA_TIMESTAMP").desc(),
                F.col("TCPM_UPDATE_DT").desc(),
                F.col("TCPM_ID").desc()
            )
        ))
        .filter(F.col("rn") == 1)
        .select(
            "TCPM_MarketCode", "TCPM_BrandCode", "TCPM_SourceSystemCode", "TCPM_TouchPointCode",
            "tcpm_distributionchannelcode", "tcpm_touchpointtypecode"
        )
        .filter(F.trim(F.coalesce(F.col("tcpm_distributionchannelcode"), F.lit(""))) != F.lit(""))
    )
    touchpoint_master_df.cache()
    print_log(f"touchpoint_master_df count: {touchpoint_master_df.count()}")


    # 取当前task_id下的所有的uid
    filer_consumer = (spark.table(consumer_table).filter(F.col("task_id") == task_id).alias("con")
        .select("scon_mrkt_code", "consumermdmkey").distinct()
        # .join(F.broadcast(touchpoint_master_df).alias("t"), 
        #     [F.col("scon_registration_toch_code") == F.col("tcpm_touchpointcode"),
        #      F.col("scon_mrkt_code") == F.col("tcpm_marketcode"),
        #      F.col("scon_brnd_code") == F.col("tcpm_brandcode")], 
        #     "left"
        # )
        # .select("con.*", F.col("t.tcpm_distributionchannelcode"))
        # .select(*group_cols).distinct()
    )


    full_consumer = (spark.table(consumer_table).alias("con")
        .join(F.broadcast(touchpoint_master_df).alias("t"), 
            [F.col("scon_registration_toch_code") == F.col("tcpm_touchpointcode"),
             F.col("SCON_SRCS_CODE") == F.col("TCPM_SourceSystemCode"),
             F.col("scon_mrkt_code") == F.col("tcpm_marketcode"),
             F.col("scon_brnd_code") == F.col("tcpm_brandcode")], 
            "left"
        )
        .select("con.*", F.col("t.tcpm_distributionchannelcode"), F.col("t.tcpm_touchpointtypecode"))
    )

    # 使用 left_semi 筛选出符合条件的行（等价于inner join 逻辑但更高效）
    join_df = full_consumer.join(F.broadcast(filer_consumer), ["scon_mrkt_code", "consumermdmkey"], "left_semi")
    
    # 先排除配置表中的 ukey，再做 source 配置关联，减少后续计算数据量
    consumer_join_sources_df = join_df \
        .join(F.broadcast(exclude_ukey_df),
              (F.col("scon_mrkt_code") == F.col("exclude_marketcode")) &
              (F.col("consumermdmkey") == F.col("exclude_consumermdmkey")),
              "left_anti") \
        .join(F.broadcast(high_count_ukey_df),
              ["scon_mrkt_code", "consumermdmkey"],
              "left_anti") \
        .join(F.broadcast(exclude_sources_df),
              (F.col("scon_mrkt_code") == F.col("tmec_marketcode")) &
              (F.col("scon_srcs_code") == F.col("tmec_sourcesystemcode")),
              "left") \
        .withColumn("is_acs_source", F.when(F.col("tmec_marketcode").isNotNull() & (F.col("tmec_type") == "ACS"), 1).otherwise(0)) \
        .drop("tmec_id","tmec_marketcode","tmec_type","tmec_sourcesystemcode") \
    
    touchpoint_master_df.unpersist()

    return consumer_join_sources_df

In [0]:
def get_all_acs_consumer_data():
    acs_sources_df = spark.table(dim_excludesource_table).filter(F.col("tmec_type") == "ACS")

    acs_consumer = (spark.table(consumer_table)
        .join(acs_sources_df,
            (F.col("scon_mrkt_code") == F.col("tmec_marketcode")) & (F.col("scon_srcs_code") == F.col("tmec_sourcesystemcode")),
            "inner"
        )
        .select(
            F.col("scon_id"),
            F.col("scon_mrkt_code"), 
            F.col("scon_brnd_code"),
            F.col("scon_srcs_code")
        ))

    return acs_consumer
